# 批量问答：原始 PDF → Markdown → 建库 → 回答问题集 → 整理保存

运行本 Notebook（从上到下依次执行所有 cell）即可完成：

1. 把一批 **原始 PDF** 转成带页码标记（`<!-- page: N -->`）的 Markdown，中间产物落盘、可复用；
2. 用 `superindex.md_ingest.index_markdown` 建库；
3. 读取 **JSON / JSONL 问题集**，规范化成 `superindex.batch.Question`；
4. 用本项目"原服务 agent 链路"（与 `superindex batch` 同一套 client / prefetch / `run_question`）逐题回答；
5. 整理保存：`summary.md`、`answers.csv`、`answers.xlsx`；
6. （可选，步骤 7）答案评测：用 deepeval 的 `GEval` 让裁判模型对照 `expected` 逐题判定答案是否正确，输出准确率（`RUN_EVAL=False` 关闭；没装 deepeval 等情况会自动跳过）。

运行到最前面会先**更新代码**（见「更新代码（git pull）」cell）：在仓库里就 `git pull --ff-only`；不在仓库里（如 Databricks）就 clone 到 `REPO_DIR` 并优先使用那份代码。
失败（没装 git、网络不通、本地有改动等）只提示不中断；不想更新就设 `GIT_PULL = False`（或环境变量 `GIT_PULL=0`）。其余配置项 `REPO_URL`、`REPO_BRANCH`、`REPO_DIR`（默认 `<当前目录>/SuperIndex`）同样可用同名环境变量覆盖。

可以在 **Databricks** 上直接作为启动脚本运行，也可以在**本地** Jupyter / VS Code / `nbconvert` 里运行：

| | Databricks | 本地（仓库内） |
|---|---|---|
| 安装 | 运行「环境安装」cell（`%pip install` + 重启 Python；`%pip install deepeval` 那格是步骤 7 的可选依赖） | `uv sync --group notebook`；**不要运行**「环境安装」cell；要评测（步骤 7）就用 `uv run --group notebook --with deepeval ...` 启动 |
| 模型 | 「模型配置」cell 里填 `LLM_CONFIG`（OpenAI 兼容端点填 `OPENAI_MODEL` / `OPENAI_BASE_URL` / `OPENAI_API_KEY`），密钥从 `dbutils.secrets` 读 | 仓库根目录 `.env`（也可用「模型配置」cell） |
| 数据 | `DATASET_PATH`、`PDF_DIR` 指向 `/Volumes/...` 之类的路径 | 本地路径 |
| 中间产物 | 与本地一致，默认 `<ROOT_DIR>/data/notebook/<题集名>/`（环境变量 `NB_WORK_DIR` 可改到集群本地盘，可写进 `.env`）；`PERSIST_DIR` 指向持久位置后自动同步 | `<ROOT_DIR>/data/notebook/<题集名>/` |

**在 Databricks 上需要填的配置**

1. **安装源**：「环境安装」cell 里默认 `%pip install superindex==0.1.2 ...`（PyPI）；公司内网请换成内部 PyPI 镜像 / Git URL / Repos 路径 / Volumes 里的 wheel（cell 注释里各有一行写法）。
2. **模型配置**：「模型配置」cell 里的 `LLM_CONFIG`（OpenAI 兼容端点 / Databricks Model Serving / Azure OpenAI / 公司 SDK 四种写法；OpenAI 兼容端点优先用 `OPENAI_MODEL` / `OPENAI_BASE_URL` / `OPENAI_API_KEY`）；网关需要自定义请求头时填 `LLM_EXTRA_HEADERS`。
3. **`DATASET_PATH`、`PDF_DIR`**：题集与 PDF 目录，如 `/Volumes/<catalog>/<schema>/<volume>/...`（`DATASET_PATH` 可写在 `.env` 的 `DATASET_PATH=...`；其它配置项同理可用同名环境变量）。`PDF_DIR` 是**递归读取**的目录：子目录里的 `*.pdf` 也会处理。
4. **`PERSIST_DIR`**：如 `/Volumes/<catalog>/<schema>/<volume>/superindex_nb/<题集名>`；不填则不同步（`WORK_DIR` 设在集群本地盘时，集群重启即丢）。

5. **更新代码**（可选）：「更新代码（git pull）」cell 默认从 GitHub clone / pull；公司内网访问不了 GitHub 时把 `REPO_URL` 改成内部镜像，或设 `GIT_PULL = False` 关掉；`REPO_DIR` 所在目录要可写。

Python 要求 **3.11+**（`superindex` 的 `requires-python`），按 Databricks 官方的 Runtime 与 Python 版本对应关系，对应 Databricks Runtime 15.x 及以上；14.x 自带 Python 3.10，装不上（该对应关系依据官方文档，我们未实测）。

**前置条件**

- 本地：在仓库根目录的 `.env` 里配置问答模型，至少 `SUPERINDEX_CHAT_MODEL`（以及对应的 base url / key，参照 `.env.example`）。**模型必须支持 tool calling。**
  只做步骤 1-3（转换、建库、读题）不需要模型；`INDEX_SUMMARY=True` 时还需要 `SUPERINDEX_INDEX_MODEL`。
- OpenAI 兼容端点也可以只写 `OPENAI_MODEL`（端点上的模型名，自动加 `openai/` 前缀）、`OPENAI_BASE_URL`（一般带 `/v1`）、`OPENAI_API_KEY` 三项（`.env` 或 `LLM_CONFIG` 里都行）：
  **只有 `OPENAI_MODEL` 非空才启用**，单独设了 `OPENAI_API_KEY` / `OPENAI_BASE_URL` 不会改变行为；对应的 `SUPERINDEX_*` 已设时以 `SUPERINDEX_*` 为准（逐变量判断）；
  建库摘要的模型（`INDEX_SUMMARY=True`）在没设 `SUPERINDEX_INDEX_MODEL` 时同用 `OPENAI_MODEL`。「环境」cell 会自己做这组映射，所以 Databricks 上装的 PyPI 版不必支持它。
- 本地依赖：`uv sync --group notebook`（`ipykernel`、`pandas`、`openpyxl`、`pycryptodome` 不在主依赖里）。Databricks 上由「环境安装」cell 安装。
  其中 `pycryptodome` 用于读取 **AES 加密的 PDF**；缺它时这类 PDF 会在步骤 1 标为 `failed` 并提示安装，其它文档照常处理。
- 只改**环境安装 / 模型配置 / 配置**这几个 cell（步骤 7 评测的配置在它自己的 cell 里），其余 cell 无需改动；所有 cell 可重复运行（已转换的 PDF、已建好的库、已成功的题都会跳过）。

**带密码的 PDF**（需要密码才能打开的 PDF）

- 主密码写在 `.env` 里的 `PDF_INGEST_PASSWORD`（如 `PDF_INGEST_PASSWORD='你的密码'`，含 `#`、空格等字符时用单引号括起来），它总是第一个被尝试。
- 其它候选密码配 `PDF_PASSWORDS`（配置 cell），排在主密码之后按顺序依次尝试。**不要把明文密码写进 Notebook**：Databricks 上写 `PDF_PASSWORDS = [dbutils.secrets.get(scope="<scope>", key="<key>")]`；
  命令行可用环境变量 `PDF_PASSWORDS`，值是 **JSON 数组字符串**（如 `'["pw1", "pw,with,comma"]'`，密码里可含逗号等任意字符）；值不是 JSON 数组时，整个字符串当作单个密码。
- 不加密、或只设了 owner 密码（打开密码为空）的 PDF 不受影响。需要密码的 PDF 用候选密码依次尝试；密码不对（或没配密码）的**跳过并标记**，不中断其它文档。
- 三种"被跳过"的状态：
  - `skipped_no_text_layer`：PDF 没有文字层（扫描件、纯图片）；已标记则下次不再重复抽取，`FORCE_EXTRACT=True` 才重试；
  - `skipped_wrong_password`：需要密码但没配密码，或候选密码都不对；**每次运行都会重新尝试**（密码配置可能已改对）；
  - `failed`：其它原因抽取失败（如 AES 加密的 PDF 缺 `pycryptodome`），不属于"跳过"。
- 解密只在**内存**里进行（文字层用带密码的 PyPDF2 reader 直接读，补标题用内存里的解密版 PDF），**不会生成解密副本文件**；落盘的只有抽取出的 Markdown 与 `meta.json`（只记 `encrypted` / `password_used` 两个布尔值，不记密码）。
  **注意：Markdown 是明文**，`WORK_DIR` / `PERSIST_DIR` 的访问权限要按原 PDF 的密级管理。
- 需要密码的 PDF 不支持页面截图（`page image`，默认关闭）：步骤 2 不对它们传 `pdf=` 关联。仅 `text-layer` 支持密码；`azure-di` 不处理带密码的 PDF。

**命令行执行（本地）**：必须带上跳过「环境安装」cell 的参数，否则 `%pip install` 会把 PyPI 上的 `superindex` 装进当前环境、覆盖本地的可编辑安装：

```bash
uv run --group notebook jupyter nbconvert --to notebook --execute notebooks/batch_qa.ipynb \
  --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags databricks-setup \
  --output-dir <某目录>
```

（配置项可用同名环境变量覆盖，也可写进 `ROOT_DIR/.env`，见配置 cell。要评测（步骤 7）就把开头换成 `uv run --group notebook --with deepeval jupyter nbconvert ...`：deepeval 不进项目依赖，`--with` 只临时叠加、不改 `uv.lock`；不加则步骤 7 自动跳过。）

**文字层转出的落盘格式**（`PDF_EXTRACTOR="text-layer"`，默认）

- 每个 PDF 存成 `WORK_DIR/md/<PDF 文件名 stem>.md`：UTF-8 Markdown，**每页前有 `<!-- page: N -->` 页码标记**；`ADD_HEADINGS=True` 时在对应页的页码标记后带 `#` 章节标题。
- 旁边的 `<stem>.meta.json` 记录：来源 PDF、抽取方式、页数、是否补了标题、空白页（数量与页码）、时间、`status`、`encrypted` / `password_used`（是否加密、是否用了密码，不记密码本身）。
- 这些 Markdown 可以**直接人工查看、修改**；改动后重新运行，步骤 2 会因内容变化自动重新建库（不用 `FORCE_EXTRACT`，那会用 PDF 重新覆盖你的修改）。
- **没有文字层**的 PDF（所有页文字都为空，如扫描件）不写 `.md`、不建库，只写 `meta.json`（`status: "skipped_no_text_layer"`）并登记到 `skipped_pdfs.json`；指向它的题目不会去问模型，而是在结果里记一条 `status=skipped_no_text_layer` 的错误记录。需要处理这类 PDF 请改用 `azure-di`。
- 默认**绝不调用 Azure**：即使环境里有 Azure 密钥，也只有显式设 `PDF_EXTRACTOR = "azure-di"` 才会用。

**产出物**（所有环境默认都在 `<ROOT_DIR>/data/notebook/<问题集文件名>/` 下，`data/notebook/` 已被 git 忽略；PDF 目录按只读处理，绝不往里写；`ROOT_DIR` 是「环境」cell 打印的绝对路径（仓库根，找不到时为当前工作目录，环境变量 `ROOT_DIR` 可覆盖），相对路径配置都以它为基准；想放集群本地盘就设环境变量 `NB_WORK_DIR`（可写进 `.env`）为本地盘上的绝对路径，用 `PERSIST_DIR` 同步到持久位置）

| 路径 | 内容 |
|---|---|
| `md/` | PDF 转出的 Markdown 及同名 `*.meta.json`，格式见下 |
| `skipped_pdfs.json` | 被跳过的 PDF（无文字层 / 密码不正确或未配置）及原因 |
| `store/` | 索引库 |
| `runs/<时间戳>/results.jsonl` | 逐题完整记录，与 `superindex batch` 格式兼容 |
| `runs/<时间戳>/summary.md`、`answers.csv`、`answers.xlsx` | 总览、答案表（csv 为 utf-8-sig，Excel 可直接打开） |
| `<ROOT_DIR>/data/eval/<问题集文件名>/eval_results.jsonl`（不在上述目录下） | 步骤 7 逐题评测记录（得分、是否通过、理由、`eval_status`），断点续跑用 |
| 同目录 `eval_summary.md`、`eval.csv`、`eval.xlsx` | 步骤 7 的准确率与逐题评测表 |

## 更新代码（git pull）

运行到这里先把项目代码更新到最新（**只用标准库，必须放在任何 `import superindex` 之前**）：

- **在仓库里运行**（从当前目录向上能找到 `pyproject.toml` + `superindex/`）：对该仓库执行 `git pull --ff-only`（用当前分支已配置的上游，只做快进，不会产生合并提交）。
- **不在仓库里**（如 Databricks 上单独导入了本 Notebook）：`REPO_DIR` 不存在就 `git clone` 到那里；已经是一份仓库副本就 `git pull --ff-only`；已存在但不是 git 仓库则提示并跳过，**不会动它**。
  clone 出来后，下面「环境」cell 会优先用这份源码（插到 `sys.path` 最前面，优先于 `%pip` 装的 PyPI 版；依赖仍由 `%pip` 安装），并在日志里打印 `superindex` 实际来自哪里；不会 `chdir`；此时 `ROOT_DIR` 即 clone 目录，`.env` 从 `ROOT_DIR/.env` 读。
- **任何失败都不中断**（没装 git、网络不通、认证失败、有本地改动无法快进、超时）：只打印原因和处理建议，然后用磁盘上现有的代码继续运行。**绝不自动丢弃或 stash 你的本地改动。**
  常见情形：在 Jupyter 里运行并保存过本 Notebook（带了输出）会产生本地改动，而上游又改了这个文件时 pull 会失败——按提示 `git -C <仓库根> stash`，或 `git -C <仓库根> checkout -- notebooks/batch_qa.ipynb` 后重跑。
- 打印时会把 `REPO_URL` 里的 `用户名:token@` 打码；`git` 调用关闭了交互式输入凭据（`GIT_TERMINAL_PROMPT=0`），clone 超时 300 秒、pull 超时 120 秒。

配置（都在下面 cell 顶部，也可用环境变量覆盖）：

| 配置项 | 环境变量 | 默认 | 含义 |
|---|---|---|---|
| `GIT_PULL` | `GIT_PULL` | `True` | `0` / `false` / `no` / `off` 关闭，此时不执行任何 git 命令 |
| `REPO_URL` | `REPO_URL` | `https://github.com/VoldemortGin/SuperIndex.git` | clone 的地址；公司内网有镜像时改这里 |
| `REPO_BRANCH` | `REPO_BRANCH` | `main` | clone 的分支（pull 用本地分支已配置的上游） |
| `REPO_DIR` | `REPO_DIR` | `<当前目录>/SuperIndex` | 不在仓库里时 clone 到哪里 |

注意：pull 更新的是**磁盘上的代码**。如果本 Notebook 文件自身被更新了，当前已打开的这份不会自动变，需要重新打开 Notebook 再运行才是新版；已经 `import` 过的 `superindex` 模块也不会自动重载（所以这一格要在最前面）。


In [ ]:
# ───────────── 更新代码（git pull）：只用标准库，必须在任何 import superindex 之前 ─────────────
# 此时还没加载 .env、也没有下面的 _env 辅助函数，所以直接读 os.environ。任何失败都只打印提示，不中断运行。
import os
import re
import shutil
import subprocess
from pathlib import Path

GIT_PULL = os.environ.get("GIT_PULL", "1").strip().lower() not in {"0", "false", "no", "off"}  # 总开关
REPO_URL = os.environ.get("REPO_URL") or "https://github.com/VoldemortGin/SuperIndex.git"  # 公司内网有镜像时改这里
REPO_BRANCH = os.environ.get("REPO_BRANCH") or "main"
# 不在仓库里时 clone 到哪里。下面"环境"cell 会用同样的规则重新算一遍（Databricks 的 restartPython 会清空变量）
REPO_DIR = Path(os.environ.get("REPO_DIR") or Path.cwd() / "SuperIndex").expanduser().resolve()

_GIT_ENV = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GCM_INTERACTIVE": "never"}  # 不要卡在交互式输入凭据


def _mask(text: object) -> str:
    """把 URL 里的 user:token@ 打码，避免凭据出现在输出里。"""
    return re.sub(r"(://)[^/@\s]+@", r"\1***@", str(text))


def _git(*args: str, timeout: int = 120) -> tuple[int, str]:
    """运行 git，返回 (退出码, 打码后的 stdout+stderr)；没装 git / 超时 抛 RuntimeError（带中文原因）。"""
    try:
        r = subprocess.run(["git", *args], capture_output=True, text=True, encoding="utf-8", errors="replace",
                           timeout=timeout, env=_GIT_ENV, stdin=subprocess.DEVNULL)
    except FileNotFoundError:
        raise RuntimeError("找不到 git 命令（本机 / 集群没装 git，或不在 PATH 里）") from None
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"git {args[0]} 超过 {timeout} 秒没完成（网络很慢或不通）") from None
    return r.returncode, _mask((r.stdout + r.stderr).strip())


def _is_repo_root(folder: Path) -> bool:  # 判定条件与"环境"cell 的 find_repo_root 一致
    return (folder / "pyproject.toml").is_file() and (folder / "superindex").is_dir()


def _is_own_worktree(root: Path) -> bool:
    """root 本身是 git 工作区的顶层（而不是碰巧落在别的 git 仓库里）。"""
    rc, out = _git("-C", str(root), "rev-parse", "--show-toplevel", timeout=30)
    if rc != 0:
        print(f"  {root} 不是 git 工作区（例如下载的 zip），跳过 git pull。" + (f" git 输出：{out}" if out else ""))
        return False
    if Path(out).resolve() != root.resolve():
        print(f"  {root} 位于另一个 git 仓库（{out}）里，不是它自己的工作区，跳过 git pull（避免误更新别的仓库）。")
        return False
    return True


def _pull(root: Path) -> None:
    rc, upstream = _git("-C", str(root), "rev-parse", "--abbrev-ref", "--symbolic-full-name", "@{u}", timeout=30)
    if rc != 0:
        print("  当前分支没有配置上游分支（或处于游离 HEAD），跳过 git pull。"
              f"需要时手动设置：git -C \"{root}\" branch --set-upstream-to=origin/<分支名>")
        return
    before = _git("-C", str(root), "rev-parse", "HEAD", timeout=30)[1]
    rc, out = _git("-C", str(root), "pull", "--ff-only", timeout=120)
    if rc != 0:
        print("  git pull 失败（已忽略，继续用现有代码运行）：\n    " + out.replace("\n", "\n    "))
        low = out.lower()
        if "overwritten" in low or "local changes" in low or "uncommitted" in low:
            print("  原因：本地有未提交的改动，且上游改了同一个文件。常见情况：在 Jupyter 里运行并保存过本 Notebook（带了输出）。\n"
                  "  本 Notebook 不会自动丢弃你的改动。处理办法（二选一，再重新运行）：\n"
                  f"    git -C \"{root}\" stash\n"
                  f"    git -C \"{root}\" checkout -- notebooks/batch_qa.ipynb   # 只还原本 Notebook")
        elif "fast-forward" in low or "diverg" in low:
            print(f"  原因：本地分支有上游没有的提交，无法快进。请手动处理：git -C \"{root}\" status")
        else:
            print("  请检查：网络能否访问 REPO_URL 对应的 git 服务器、是否需要认证（本 Notebook 不会弹出输入凭据）。")
        return
    after = _git("-C", str(root), "rev-parse", "HEAD", timeout=30)[1]
    if before == after:
        print(f"  已是最新（{after[:7]}，上游 {upstream}）")
    else:
        n = _git("-C", str(root), "rev-list", "--count", f"{before}..{after}", timeout=30)[1]
        print(f"  已更新：从 {before[:7]} 更新到 {after[:7]}，{n} 个提交（上游 {upstream}）")
        print("  提示：已经 import 过的 superindex 不会自动重载；本 Notebook 文件若也被更新，请重新打开再运行。")


def _clone() -> None:
    created = not REPO_DIR.exists()
    ok = False
    try:
        print(f"  clone {_mask(REPO_URL)}（分支 {REPO_BRANCH}）→ {REPO_DIR}")
        rc, out = _git("clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR), timeout=300)
        ok = rc == 0
        if ok:
            head = _git("-C", str(REPO_DIR), "rev-parse", "--short", "HEAD", timeout=30)[1]
            print(f"  已 clone（{head}）。下面「环境」cell 会优先使用这份代码。")
        else:
            print("  git clone 失败（已忽略，继续用已安装的 superindex）：\n    " + out.replace("\n", "\n    "))
            print("  请检查：REPO_URL / REPO_BRANCH 是否正确、网络能否访问、是否需要认证、REPO_DIR 所在目录是否可写。")
    finally:
        if not ok and created and REPO_DIR.exists():
            shutil.rmtree(REPO_DIR, ignore_errors=True)  # 清掉本次失败（含超时）留下的半成品；不碰原本就存在的目录


def update_code() -> None:
    if not GIT_PULL:
        print("更新代码：已关闭（GIT_PULL=False / GIT_PULL=0），不执行 git 命令，沿用磁盘上现有代码。")
        return
    start = Path.cwd().resolve()
    root = next((f for f in (start, *start.parents) if _is_repo_root(f)), None)
    if root is not None:
        print(f"更新代码：在仓库里（{root}），git pull --ff-only")
        if _is_own_worktree(root):
            _pull(root)
    elif not REPO_DIR.exists() or (REPO_DIR.is_dir() and not any(REPO_DIR.iterdir())):
        print(f"更新代码：当前不在仓库里，{REPO_DIR} 还不存在，git clone")
        _clone()
    elif REPO_DIR.is_dir() and _is_repo_root(REPO_DIR):
        print(f"更新代码：当前不在仓库里，{REPO_DIR} 已是一份仓库副本，git pull --ff-only")
        if _is_own_worktree(REPO_DIR):
            _pull(REPO_DIR)
    else:
        print(f"更新代码：{REPO_DIR} 已存在但不是 SuperIndex 仓库，跳过（不会动它）。换个目录请设 REPO_DIR。")


try:
    update_code()
except Exception as _exc:  # noqa: BLE001 - 更新代码是锦上添花，任何意外都不能中断后面的运行
    print(f"更新代码：出错（已忽略，继续用现有代码运行）：{type(_exc).__name__}: {_mask(_exc)}")

## 环境安装（仅 Databricks）

**只在 Databricks 上运行下面三个 cell**（已打上 cell tag `databricks-setup`）。本地 Jupyter 已经 `uv sync --group notebook` 的，**不要运行**这几个 cell：`%pip install` 会把 PyPI 上的 `superindex` 装进当前环境，覆盖掉本地的可编辑安装。

- 第一个 cell 只放 `%pip install`（Databricks 要求 `%pip` 写在 cell 开头，才是 notebook 作用域安装）。默认从 PyPI 装固定版本 `superindex==0.1.2`；
  集群访问不了公网 PyPI 时，把那一行整行换成下面某一种备选写法（cell 里只保留一行 `%pip`，不要加注释行）。
  `pandas` / `openpyxl` 集群一般已预装，写在命令里只是保证有；本 Notebook 兼容旧版 pandas，不必升级它。
- 第二个 cell `%pip install deepeval` 是**步骤 7（答案评测）的可选依赖**。注意：deepeval 要求 `click<8.4`，会把集群当前 notebook 环境里的 `click` 降到 8.4 以下（`huggingface-hub` 也可能随之降级；superindex 本身不用 `click`）。不需要评测就删掉 / 跳过这个 cell，步骤 7 会自动跳过。
- 第三个 cell 重启 Python，让新装的包生效；只在 Databricks 上才会真的重启，其它环境只打印一句提示。重启会清空内存里的变量，所以它必须在后面所有配置 cell 之前运行。

依赖由 `%pip` 安装；如果前面的「更新代码」cell 在 `REPO_DIR` 里 clone 了仓库，代码本身用仓库里的最新版（「环境」cell 把它插到 `sys.path` 最前面），优先于 `%pip` 装的 PyPI 版。

备选安装写法（任选其一，替换第一个 cell 的内容）：

```
%pip install --index-url https://<公司内部 PyPI 镜像>/simple superindex==0.1.2 pandas openpyxl pycryptodome
%pip install git+https://<git 服务器>/<组织>/SuperIndex.git@<tag 或 commit> pandas openpyxl pycryptodome
%pip install /Workspace/Repos/<用户>/SuperIndex pandas openpyxl pycryptodome
%pip install /Volumes/<catalog>/<schema>/<volume>/wheels/superindex-0.1.2-py3-none-any.whl pandas openpyxl pycryptodome
```

命令行用 `nbconvert` 在本地执行时，用下面的参数跳过这几个 cell：

```bash
uv run --group notebook jupyter nbconvert --to notebook --execute notebooks/batch_qa.ipynb \
  --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags databricks-setup \
  --output-dir <某目录>
```

In [ ]:
%pip install superindex==0.1.2 pandas openpyxl pycryptodome

In [ ]:
%pip install deepeval

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

In [ ]:
# ───────────── 模型配置（必须在任何 import superindex / litellm 之前运行）─────────────
# 本 cell 只写环境变量，不 import litellm / superindex，所以下面这些变量能赶在它们之前生效。
# 默认不改动任何环境变量：本地有 .env 时行为不变（.env 在下一格"环境"里才加载，且不覆盖这里已设的变量）。
import os

os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")  # litellm 用内置价格表，import 时不联网拉取
os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")    # 关闭 openai-agents 的 tracing 上报

# 想改哪项就取消注释并填值；值为空串的项会被跳过。**不要把密钥明文写进 Notebook**。
# 在 Databricks 上从 secrets 读：dbutils.secrets.get(scope="<scope>", key="<key>")
LLM_CONFIG: dict[str, str] = {
    # ── 方式一：OpenAI 兼容端点（vLLM、公司网关等）：端点上的模型名（自动加 openai/ 前缀），地址一般带 /v1 ──
    # 只有 OPENAI_MODEL 非空才启用这一组；对应的 SUPERINDEX_* 已设（非空）时以 SUPERINDEX_* 为准。映射在下一格"环境"里做。
    # "OPENAI_MODEL": "<模型名>",
    # "OPENAI_BASE_URL": "https://<公司网关>/v1",
    # "OPENAI_API_KEY": dbutils.secrets.get(scope="<scope>", key="<key>"),
    # 等价的 SUPERINDEX_* 写法（也可以用）：
    # "SUPERINDEX_CHAT_MODEL": "openai/<模型名>",
    # "SUPERINDEX_BASE_URL": "https://<公司网关>/v1",
    # "SUPERINDEX_API_KEY_OVERRIDE": dbutils.secrets.get(scope="<scope>", key="<key>"),
    #
    # ── 方式二：Databricks Model Serving：databricks/<serving endpoint 名> ──
    # "SUPERINDEX_CHAT_MODEL": "databricks/<endpoint 名>",
    # "SUPERINDEX_BASE_URL": "https://<workspace 域名>/serving-endpoints",
    # "SUPERINDEX_API_KEY_OVERRIDE": dbutils.secrets.get(scope="<scope>", key="<key>"),  # Databricks token
    #
    # ── 方式三：Azure OpenAI：azure/<部署名> ──
    # "SUPERINDEX_CHAT_MODEL": "azure/<deployment 名>",
    # "AZURE_API_BASE": "https://<资源名>.openai.azure.com",
    # "AZURE_API_KEY": dbutils.secrets.get(scope="<scope>", key="<key>"),
    # "AZURE_API_VERSION": "<api 版本>",
    #
    # ── 方式四：公司自封装的 Python SDK：模型名写 company/<任意名>，并启用下面的"公司 SDK 适配器"cell ──
    # "SUPERINDEX_CHAT_MODEL": "company/<任意名>",
    #
    # ── 其它 ──
    # "SUPERINDEX_INDEX_MODEL": "",         # 只在 INDEX_SUMMARY=True 时用到；写法同 SUPERINDEX_CHAT_MODEL；用 OPENAI_MODEL 时不设则同用它
    # "SUPERINDEX_REASONING_EFFORT": "",    # 公司模型一般留空；设了而模型不支持会报 UnsupportedParamsError
}

# 公司网关要求的自定义请求头（如 {"X-Gateway-Token": dbutils.secrets.get(...)}）。默认空 = 不加。
# 只作用于问答（步骤 4）；建库摘要（INDEX_SUMMARY=True）这条路径不带这些 header。
LLM_EXTRA_HEADERS: dict[str, str] = {}

for _name, _value in LLM_CONFIG.items():
    if _value:
        os.environ[_name] = _value
print("模型配置：写入环境变量 " + (", ".join(k for k, v in LLM_CONFIG.items() if v) or "（无，沿用 .env / 进程环境变量）"))
print("自定义请求头：" + (", ".join(LLM_EXTRA_HEADERS) or "（无）"))

In [ ]:
# 确定唯一的绝对路径基准 ROOT_DIR（.env、data/、相对路径配置都以它为基准，不依赖 cwd、不 chdir）。必须在 import superindex 之前做。
# 顺序：环境变量 ROOT_DIR > 从 cwd 向上找到的仓库根 > clone 副本（REPO_DIR）> 当前工作目录。
import io
import json
import os
import re
import shutil
import sys
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
import pandas as pd
from IPython.display import display

try:
    from tqdm import tqdm
except ImportError:  # 没有 tqdm 就退化为 print
    tqdm = None


def find_repo_root(start: Path) -> Path | None:
    """向上找含 pyproject.toml 和 superindex/ 包目录的目录（即本仓库根）；找不到（pip 安装、不在仓库里）返回 None。"""
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "superindex").is_dir():
            return folder
    return None


IS_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))
_repo_root = find_repo_root(Path.cwd().resolve())
# 不在仓库里时，看「更新代码」cell clone 出来的副本（REPO_DIR）。默认值规则与那一格一致，这里重新算一遍：
# Databricks 的 restartPython 会清空变量，不能依赖那一格里的 Python 变量。
CLONE_DIR = Path(os.environ.get("REPO_DIR") or Path.cwd() / "SuperIndex").expanduser().resolve()
CLONE_ROOT = CLONE_DIR if _repo_root is None and (CLONE_DIR / "pyproject.toml").is_file() and (CLONE_DIR / "superindex").is_dir() else None
ROOT_DIR = Path(os.environ.get("ROOT_DIR", "").strip() or _repo_root or CLONE_ROOT or Path.cwd()).expanduser().resolve()
if _repo_root is not None or CLONE_ROOT is not None:
    sys.path.insert(0, str(_repo_root or CLONE_ROOT))  # import superindex 用这份源码，而不是 %pip 装的 PyPI 版；不 chdir


def under_root(p: str | Path) -> Path:
    """绝对路径原样（resolve）；相对路径以 ROOT_DIR 为基准。"""
    p = Path(p).expanduser()
    return (p if p.is_absolute() else ROOT_DIR / p).resolve()


print(f"ROOT_DIR : {ROOT_DIR}")

from superindex.runtime import (
    ConfigError,
    LLMSettings,
    configure_litellm,
    getenv,
    resolve_instructions,
    set_offline_defaults,
)

set_offline_defaults()  # 必须早于任何 litellm 的 import
ENV_FILES = [ROOT_DIR / ".env"] if (ROOT_DIR / ".env").is_file() else []
for _env_file in ENV_FILES:
    load_dotenv(_env_file, override=False, encoding="utf-8")  # 读取 ROOT_DIR/.env；已设置的环境变量优先


def apply_openai_env() -> dict[str, str]:
    """OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY -> SUPERINDEX_*（与新版 superindex.runtime 同一规则；
    这里自己做一遍，PyPI 旧版没有该回退时行为也一致）。只有 OPENAI_MODEL 非空才启用；
    SUPERINDEX_*（含旧名 PAGEINDEX_*）已设为非空的不覆盖。返回实际写入的 {变量名: 值}。"""
    model = os.environ.get("OPENAI_MODEL", "").strip()
    if not model:
        return {}
    model = model if model.startswith("openai/") else f"openai/{model}"  # 模型名含 "/" 也要加，别让 litellm 把前半当 provider
    wanted = {
        "SUPERINDEX_CHAT_MODEL": model,
        "SUPERINDEX_INDEX_MODEL": model,  # 建库摘要（INDEX_SUMMARY=True）同用这个模型
        "SUPERINDEX_BASE_URL": os.environ.get("OPENAI_BASE_URL", "").strip(),
        "SUPERINDEX_API_KEY_OVERRIDE": os.environ.get("OPENAI_API_KEY", "").strip(),
    }
    applied = {}
    for name, value in wanted.items():
        if value and not (getenv(name) or "").strip():
            os.environ[name] = applied[name] = value
    return applied


_applied = apply_openai_env()
if os.environ.get("OPENAI_MODEL", "").strip():
    print("模型配置：使用 OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY"
          f"（模型 {getenv('SUPERINDEX_CHAT_MODEL')}，地址 {getenv('SUPERINDEX_BASE_URL') or '（未设）'}；"
          f"已填入 {', '.join(_applied) or '（无，SUPERINDEX_* 均已设置）'}）")  # 不打印 key

from superindex import page_images, prefetch
from superindex.batch import (
    RESULTS_FILE,
    SUMMARY_FILE,
    Question,
    _from_item,
    _scope,
    judge_hits,
    read_results,
    run_question,
    score,
    write_summary,
)
from superindex.cli import make_client
from superindex.engine.local_store import DocStore
from superindex.extractors.backend import Extractor
from superindex.md_ingest import PAGE_MARKER_RE, index_markdown

print(f"运行环境: {'Databricks' if IS_DATABRICKS else '本地'}")
print(f"仓库根目录: {_repo_root}" if _repo_root else "未从当前工作目录向上找到仓库根目录")
if CLONE_ROOT is not None:
    print(f"代码来源: clone 副本 {CLONE_ROOT}（优先于 pip 安装的版本；不切换目录）")


def _superindex_version(src: Path) -> str:
    try:
        if (src.parent / "pyproject.toml").is_file():  # 源码树：读 pyproject 里的版本（已安装包的元数据可能是别的副本的）
            import tomllib

            return tomllib.loads((src.parent / "pyproject.toml").read_text(encoding="utf-8"))["project"]["version"]
        from importlib.metadata import version

        return version("superindex")
    except Exception:  # noqa: BLE001
        return "未知"


import superindex

_SRC = Path(superindex.__file__).resolve().parent
print(f"superindex 来自: {_SRC}（版本 {_superindex_version(_SRC)}）")
print(f".env      : {', '.join(map(str, ENV_FILES)) or '（未找到，仅使用进程环境变量）'}")

In [ ]:
# ───────────── 配置（所有可调项都在这一格）─────────────
# 每项都可用同名环境变量覆盖（方便命令行 nbconvert 执行；也可写进 ROOT_DIR/.env），例如 LIMIT=3。
# 例外：工作目录 WORK_DIR 的环境变量叫 NB_WORK_DIR（见下）。
def _env(name: str, default: Any, cast=str) -> Any:
    raw = os.environ.get(name)
    return default if raw is None or raw.strip() == "" else cast(raw.strip())


def _bool(text: str) -> bool:
    return text.lower() in ("1", "true", "yes", "on")


# --- 输入 ---
# Databricks 上写 Volumes / DBFS 的 FUSE 路径，如 "/Volumes/<catalog>/<schema>/<volume>/qa/questions.jsonl"、
# "/dbfs/<目录>/questions.jsonl"（占位符请换成实际值）。
# 问题文件路径优先读环境变量 DATASET_PATH（写进 .env 即可：DATASET_PATH=data/golden_questions.json；上一格已加载 .env），
# 没设用默认值。相对路径以 ROOT_DIR 为基准（下面所有路径配置同理）。
DATASET_PATH = under_root(_env("DATASET_PATH", "data/questions.jsonl"))  # 问题文件：.json 或 .jsonl
# 如 "/Volumes/<catalog>/<schema>/<volume>/qa/pdfs"、"/dbfs/<目录>/pdfs"
PDF_DIR = under_root(_env("PDF_DIR", "data/pdfs"))                          # 原始 PDF 目录（递归查找 *.pdf）

# 数据集字段名映射：{规范名: 数据集里的字段名或路径}。
# expected / doc / id / pages 可缺省：值设为 None，或数据里没有该字段（含路径取不到）都行。
# doc 缺省 = 在全库范围提问。其余字段（包括被映射走的嵌套源字段，如整个 evidence）原样带进结果（Question.extra）。
#
# 路径语法（值除了顶层字段名，还可以是路径；顶层字段名本身含 "." 的极端情况不支持）：
#   "meta.source"      用 "." 逐层取嵌套对象
#   "evidence[].file"  "[]" 展开列表：取每个元素的 file，得到列表；去重、保持首次出现顺序、丢弃空值
# 取到列表时：id / question / expected 只有 1 个元素就取该元素，多个用换行拼成字符串；
#            doc 列表即多个文档（字符串沿用原行为）；
#            pages（可选，默认 None）：检索命中判定用的页码（batch 的 `pages` 字段，元素可为 12 或 "12-13"）。
# 示例（题目带 evidence 列表、每项有 file / page 的数据集）：
#   {"id": "id", "question": "question", "expected": "ground_truth",
#    "doc": "evidence[].file", "pages": "evidence[].page"}
# 注意：把 doc 映射到标准证据文档，会把检索范围限定在正确文档内，评测成绩会偏乐观；
#       想评估"全库检索"就把 doc 设为 None。pages 多文档时是各文档页码的并集（batch 只按页码判定，不区分文档）。
# 可用环境变量 FIELD_MAP 传入整个映射（JSON 字符串）。
FIELD_MAP = _env("FIELD_MAP", {"id": "id", "question": "question", "expected": "expected", "doc": "doc",
                               "pages": None}, json.loads)
# JSON 顶层是 {"questions": [...]} 或直接列表时无需设置；顶层是其它键名（如 {"items": [...]}）时填这个键名。
RECORDS_KEY: str | None = _env("RECORDS_KEY", None)

# --- PDF → Markdown ---
PDF_EXTRACTOR = _env("PDF_EXTRACTOR", "text-layer")  # "text-layer"：PDF 文字层（原始 PageIndex 做法，完全离线）；
                                                     # "azure-di"：Azure Document Intelligence（需 AZURE_DI_ENDPOINT / AZURE_DI_KEY）
ADD_HEADINGS = _env("ADD_HEADINGS", True, _bool)     # 仅 text-layer：用离线版面分析（flash）给 Markdown 补 # 章节标题
FORCE_EXTRACT = _env("FORCE_EXTRACT", False, _bool)  # True：已有 .md 也重新转换（改了 PDF_EXTRACTOR / ADD_HEADINGS 后需要）

# --- 带密码的 PDF（需要密码才能打开的 PDF；默认不处理）---
# 主密码放在环境变量 PDF_INGEST_PASSWORD 里（写进 .env 即可：PDF_INGEST_PASSWORD='你的密码'；上一格已加载 .env）。
# 它原样当作一个密码（不做 JSON 解析），总是排在候选列表第一个。.env 里密码含 #、空格、引号等字符时用单引号括起来。
# 其它候选密码填 PDF_PASSWORDS，排在主密码之后依次尝试。都没配 = 需要密码的 PDF 一律跳过并标记（不影响其它 PDF）。
# 不加密、或只设了 owner 密码（打开密码为空）的 PDF 不受影响，照常处理。
# 不要把明文密码写进 Notebook。Databricks 上没有 .env 时从 secrets 读（把下一行 [] 换成）：
#   PDF_PASSWORDS = [dbutils.secrets.get(scope="<scope>", key="<key>")]  # 多个密码就写多项
# 也可用环境变量 PDF_PASSWORDS 覆盖：值是 JSON 数组字符串，如 '["pw1", "pw,with,comma"]'（密码里可含逗号等任意字符，
# 与 FIELD_MAP 的 JSON 传法一致）；值不是 JSON 数组（不是合法 JSON，或是 JSON 数字/字符串）时，整个字符串当作单个密码。
# 密码不会被写进 meta.json / skipped_pdfs.json / 任何输出；步骤 1 只在内存里解密，不落解密副本。
PDF_PASSWORDS: list[str] = []


def parse_passwords(raw: str) -> list[str]:
    """环境变量值 -> 候选密码列表：JSON 数组按元素；其它（含非法 JSON）整个字符串当单个密码。"""
    try:
        data = json.loads(raw)
    except ValueError:
        return [raw]
    return [str(p) for p in data if p not in (None, "")] if isinstance(data, list) else [raw]


_env_passwords = os.environ.get("PDF_PASSWORDS")
if _env_passwords is not None and _env_passwords.strip():
    PDF_PASSWORDS = parse_passwords(_env_passwords)
_main_password = os.environ.get("PDF_INGEST_PASSWORD", "")
PDF_PASSWORDS = [_main_password, *PDF_PASSWORDS]  # 主密码排第一
PDF_PASSWORDS = list(dict.fromkeys(p for p in PDF_PASSWORDS if p))  # 去重保序；空串不是密码（空密码本来就会先试）

# --- 中间产物目录 ---
# 所有环境默认 ROOT_DIR/data/notebook/<题集名>（所有运行产物都放 ROOT_DIR/data/ 下）。Databricks 上想放集群本地盘（FUSE 路径不一定支持存储层用的
# flock / 原子替换 / 逐行追加），设环境变量 NB_WORK_DIR 为本地盘（local_disk0）上的绝对路径即可（写进 .env 即可：NB_WORK_DIR=/local_disk0/...；上一格已加载 .env）。
WORK_DIR = under_root(os.environ.get("NB_WORK_DIR", "").strip() or ROOT_DIR / "data" / "notebook" / DATASET_PATH.stem)
# PDF 目录按只读处理：工作目录（md/ store/ runs/）绝不能落在里面（也避免把产物当成输入 PDF 递归扫到）
if WORK_DIR.resolve().is_relative_to(PDF_DIR.resolve()):
    raise ValueError(f"工作目录落在 PDF 目录之内；PDF 目录按只读处理，请修改 NB_WORK_DIR 或 PDF_DIR。WORK_DIR={WORK_DIR}，PDF_DIR={PDF_DIR}")
# 持久目录（默认 None = 不同步）。Databricks 上建议填持久路径，如 "/Volumes/<catalog>/<schema>/<volume>/superindex_nb/<题集名>"：
# 开始时把里面已有的 md/ store/ runs/ skipped_pdfs.json 恢复到本地 WORK_DIR（集群重启后不必重抽取、重建库，已答的题可续跑），
# 步骤 2 建库后和全部完成后再把 WORK_DIR 复制回去。
_persist = _env("PERSIST_DIR", None, Path)
PERSIST_DIR: Path | None = under_root(_persist) if _persist else None
if PERSIST_DIR is not None and PERSIST_DIR.resolve().is_relative_to(PDF_DIR.resolve()):
    raise ValueError(f"持久目录落在 PDF 目录之内；PDF 目录按只读处理，请修改 PERSIST_DIR 或 PDF_DIR。PERSIST_DIR={PERSIST_DIR}，PDF_DIR={PDF_DIR}")
MD_DIR = WORK_DIR / "md"        # 转出的 Markdown
STORE = WORK_DIR / "store"      # 索引库
RUNS_DIR = WORK_DIR / "runs"    # 每次运行一个时间戳目录

# --- 建库 ---
INDEX_SUMMARY = _env("INDEX_SUMMARY", False, _bool)  # True：建库时用 LLM（SUPERINDEX_INDEX_MODEL）写节点摘要；False：不调 LLM

# --- 问答 ---
LIMIT: int | None = _env("LIMIT", None, int)    # 只跑前 N 题；None = 全部
TIMEOUT = _env("TIMEOUT", 300.0, float)         # 每题超时（秒）
CONCURRENCY = _env("CONCURRENCY", 1, int)       # 并发题数
RESUME = _env("RESUME", True, _bool)            # True：续跑最近一次运行目录（跳过已成功的题、重跑出错的题）；False：新开时间戳目录
PREFETCH: bool | None = _env("PREFETCH", None, _bool)  # 关键词检索前置：None = 沿用项目默认（SUPERINDEX_PREFETCH，默认开）
PREFETCH_K: int | None = _env("PREFETCH_K", None, int)  # 前置页数：None = 沿用项目默认（5）

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"问题集   : {DATASET_PATH}")
print(f"PDF 目录 : {PDF_DIR}")
print(f"工作目录 : {WORK_DIR}")
print(f"MD_DIR   : {MD_DIR}")
print(f"STORE    : {STORE}")
print(f"RUNS_DIR : {RUNS_DIR}")
print(f"持久目录 : {PERSIST_DIR or '（未设置，不同步）'}")
print(f"抽取方式 : {PDF_EXTRACTOR}（补标题：{ADD_HEADINGS if PDF_EXTRACTOR == 'text-layer' else '不适用'}）")
print(f"PDF 密码 : 已配置 {len(PDF_PASSWORDS)} 个候选密码")  # 只打印个数，绝不打印密码本身

In [ ]:
# ───────────── 持久化目录同步（仅当配置了 PERSIST_DIR；否则下面的函数什么都不做）─────────────
PERSIST_ITEMS = ("md", "store", "runs", "skipped_pdfs.json")  # 开始时从 PERSIST_DIR 恢复的内容


def _copy_files(src: Path, dst: Path) -> int:
    """把 src（文件或目录）复制到 dst，返回文件数。只复制内容、不复制权限/时间戳（FUSE 路径上 copystat 常失败）。"""
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dst)
        return 1
    count = 0
    for path in src.rglob("*"):
        if path.is_file():
            target = dst / path.relative_to(src)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, target)
            count += 1
    return count


def _persist_dir() -> Path | None:
    if PERSIST_DIR is None:
        return None
    if PERSIST_DIR.resolve() == WORK_DIR.resolve():
        print("PERSIST_DIR 与 WORK_DIR 相同，无需同步")
        return None
    return PERSIST_DIR


def restore_from_persist() -> None:
    """WORK_DIR 里还没有的 md/store/runs/skipped_pdfs.json，从 PERSIST_DIR 复制进来（集群重启后接着上次的进度）。"""
    persist = _persist_dir()
    if persist is None:
        return
    for name in PERSIST_ITEMS:
        src, dst = persist / name, WORK_DIR / name
        if not src.exists() or dst.exists():
            continue
        try:
            print(f"从 PERSIST_DIR 恢复 {name}（{_copy_files(src, dst)} 个文件）")
        except Exception as exc:  # noqa: BLE001 - 恢复失败只警告：删掉复制了一半的残留，该项之后重新生成
            shutil.rmtree(dst, ignore_errors=True)
            print(f"警告：从 PERSIST_DIR 恢复 {name} 失败（{type(exc).__name__}: {exc}），将重新生成")


def sync_to_persist(label: str) -> None:
    """把 WORK_DIR 的内容复制到 PERSIST_DIR；失败只警告、不中断。"""
    persist = _persist_dir()
    if persist is None:
        return
    try:
        copied = [f"{src.name}（{_copy_files(src, persist / src.name)} 个文件）"
                  for src in sorted(WORK_DIR.iterdir())] if WORK_DIR.is_dir() else []
        print(f"[{label}] 已同步到 {persist}：" + ("、".join(copied) or "（WORK_DIR 里还没有内容）"))
    except Exception as exc:  # noqa: BLE001
        print(f"警告：同步到 PERSIST_DIR 失败（{type(exc).__name__}: {exc}）；本地结果仍在 {WORK_DIR}")


restore_from_persist()

In [ ]:
# ───────────── 可选：公司自封装 Python SDK 适配器（默认关闭）─────────────
# 公司模型不是 OpenAI 协议、只能通过公司 Python SDK 调用时，把 USE_CUSTOM_LLM 改 True，
# 在 company_chat 里调用公司 SDK，并把模型名配成 company/<任意名>（上面"模型配置"的方式四；
# 问答用 SUPERINDEX_CHAT_MODEL，INDEX_SUMMARY=True 时建库摘要用 SUPERINDEX_INDEX_MODEL，同样写 company/...）。
# 模板把 company_chat 的返回值转成 litellm 需要的格式；流式是"假流式"（整段结果一次性吐出）。
# USE_CUSTOM_LLM=False 时本 cell 什么都不做（不 import litellm）。
USE_CUSTOM_LLM = _env("USE_CUSTOM_LLM", False, _bool)


def company_chat(messages: list[dict], tools: list[dict] | None) -> dict:
    """★ 在这里调用公司 SDK ★

    入参：
      messages  OpenAI 风格的消息列表：{"role": "system"/"user"/"assistant"/"tool", "content": str, ...}；
                assistant 消息可能带 "tool_calls"，tool 消息带 "tool_call_id"（图片等非文字内容已被去掉）。
      tools     OpenAI 风格的工具定义列表 [{"type": "function", "function": {"name", "description", "parameters"}}]，
                可能为 None。问答 agent 靠它调用 search_pages / get_page_content 等，模型必须能返回工具调用。
    返回：{"text": str | None, "tool_calls": [{"id": str, "name": str, "arguments": str(JSON)}]}
      要调用工具时 tool_calls 非空（text 可为 None）；给最终回答时 tool_calls 为空、text 是回答。
    """
    raise NotImplementedError("请在 company_chat 里调用公司 SDK")


def register_company_llm() -> None:
    """把 company_chat 注册成 litellm 的自定义 provider "company"。"""
    import litellm
    from litellm import CustomLLM
    from litellm.types.utils import GenericStreamingChunk

    def text_of(content: Any) -> str:
        if isinstance(content, list):  # 多段内容：只保留文字
            return "".join(p.get("text", "") for p in content if isinstance(p, dict) and p.get("type") == "text")
        return content if isinstance(content, str) else "" if content is None else json.dumps(content, ensure_ascii=False)

    def call(messages: list[dict], optional_params: dict) -> tuple[str | None, list[dict]]:
        result = company_chat([{**m, "content": text_of(m.get("content"))} for m in messages],
                              optional_params.get("tools"))
        calls = [{"id": c["id"], "type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                 for c in result.get("tool_calls") or []]
        return result.get("text"), calls

    def to_response(result: tuple[str | None, list[dict]], model_response: Any, model: str) -> Any:
        text, calls = result
        model_response.choices[0].message = litellm.Message(content=text, role="assistant", tool_calls=calls or None)
        model_response.choices[0].finish_reason = "tool_calls" if calls else "stop"
        model_response.model = model
        return model_response

    def to_chunks(result: tuple[str | None, list[dict]]) -> list[GenericStreamingChunk]:
        text, calls = result
        usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}  # SDK 有用量数据可在此填入
        chunks = [GenericStreamingChunk(text=text or "", is_finished=False, finish_reason="", usage=usage,
                                        index=0, tool_use=None)]
        chunks += [GenericStreamingChunk(text="", is_finished=False, finish_reason="", usage=usage, index=0,
                                         tool_use={**call, "index": i}) for i, call in enumerate(calls)]
        chunks[-1]["is_finished"] = True
        chunks[-1]["finish_reason"] = "tool_calls" if calls else "stop"
        return chunks

    class CompanyLLM(CustomLLM):
        def completion(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                       encoding, api_key, logging_obj, optional_params, **kwargs):
            return to_response(call(messages, optional_params), model_response, model)

        async def acompletion(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                              encoding, api_key, logging_obj, optional_params, **kwargs):
            return to_response(call(messages, optional_params), model_response, model)

        def streaming(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                      encoding, api_key, logging_obj, optional_params, **kwargs):
            yield from to_chunks(call(messages, optional_params))

        async def astreaming(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                             encoding, api_key, logging_obj, optional_params, **kwargs):
            for chunk in to_chunks(call(messages, optional_params)):
                yield chunk

    others = [p for p in (litellm.custom_provider_map or []) if p.get("provider") != "company"]
    litellm.custom_provider_map = [*others, {"provider": "company", "custom_handler": CompanyLLM()}]


if USE_CUSTOM_LLM:
    register_company_llm()
    print("已注册自定义 LLM provider：模型名写 company/<任意名>")

## 步骤 1：PDF → Markdown

逐个 PDF 转换，`md/` 下已有同名 `.md` 就跳过（`FORCE_EXTRACT=True` 强制重转）。每个 `.md` 旁写一个 `.meta.json`。

- `text-layer`：`Extractor(env={}).document_text(pdf)` 读文字层并写入 `<!-- page: N -->` 页码标记（`env={}` 保证绝不调用 Azure）。
  `ADD_HEADINGS=True` 时再用 `page_index_flash`（离线版面分析，不调 LLM）得到章节树，在对应页的页码标记之后插入 `#` 标题；
  flash 失败或得不到结构时只给警告并退回无标题版本。
  **所有页文字都为空**的 PDF（扫描件）不写 `.md`、不建库：只写 `meta.json`（`status: "skipped_no_text_layer"`）并登记到 `WORK_DIR/skipped_pdfs.json`，下次运行不再重复抽取；
  部分页为空的 PDF 照常处理，空页数量和页码记在 `meta.json` 里并在汇总表给出警告。单个 PDF 出问题都不影响其它文档。
- **带密码的 PDF**：先试空密码（只设了 owner 密码的 PDF 就这样打开，走原有流程），不行再按 `PDF_PASSWORDS` 顺序逐个试。成功后文字层用带密码的 PyPDF2 reader 直接读，
  补标题（flash 的签名不能传密码）用内存里解密好的 PDF（BytesIO）——整个过程**不生成解密副本文件**，落盘的只有 Markdown 和 `meta.json`（只记 `encrypted` / `password_used`，不记密码）。
  没配密码或候选密码都不对：不写 `.md`、不建库，`meta.json` 标 `status: "skipped_wrong_password"` 并登记到 `skipped_pdfs.json`；**下次运行会重新尝试**（密码配置可能已改对）。
  Markdown 是明文，`WORK_DIR` / `PERSIST_DIR` 的访问权限要按原 PDF 的密级管理。
- 汇总表的 `status`：`converted` 已转换 / `skipped_existing` 已有 .md 跳过 / `skipped_no_text_layer` 无文字层 / `skipped_wrong_password` 密码不正确或未配置 / `failed` 失败。
  指向被跳过 PDF 的题（步骤 3-4）不会去问模型，结果里 `status` 记为对应的 `skipped_*`。
- **AES 加密的 PDF** 需要 `pycryptodome`（`notebook` 依赖组已含）；缺失时该 PDF 标为 `failed` 并给出安装提示，其它文档继续处理。
- `azure-di`：直接复用 `extractors.azure_di.extract_corpus`（Azure 输出本身带标题和表格）。

In [ ]:
def list_pdfs(pdf_dir: Path) -> list[Path]:
    if not pdf_dir.is_dir():
        raise FileNotFoundError(f"PDF 目录不存在：{pdf_dir.resolve()}（请修改配置里的 PDF_DIR）")
    pdfs = sorted(p for p in pdf_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf")
    if not pdfs:
        raise FileNotFoundError(f"{pdf_dir.resolve()} 下没有 PDF 文件")
    return pdfs


def add_headings(md: str, pdf: Path, password: str | None = None) -> tuple[str, int, str | None]:
    """用 flash 离线版面分析给 md 补 # 标题。返回 (新 md, 标题数, 警告)；失败时原样返回 md 并给出警告。
    password 不为 None 时（PDF 需要密码）：flash 不能传密码，改给它一份内存里解密好的 PDF（BytesIO，不落盘）。"""
    try:
        from superindex.engine.flash import page_index_flash

        source = decrypted_stream(pdf, password) if password is not None else str(pdf)
        tree = page_index_flash(source, summary=False, optimize=False)
    except Exception as exc:  # noqa: BLE001 - 失败只退回无标题版本
        return md, 0, f"flash 失败（{type(exc).__name__}: {exc}），已退回无标题版本"

    heads: dict[int, list[str]] = {}

    def walk(nodes: list[dict] | None, level: int) -> None:
        for node in nodes or []:
            title = " ".join(str(node.get("title") or "").split())
            page = node.get("start_index")
            # flash 在找不到结构时会退化成一页一个 "Page N" 节点，那不是真标题
            if title and isinstance(page, int) and not re.fullmatch(r"Page \d+", title):
                heads.setdefault(page, []).append("#" * min(level, 6) + " " + title)
            walk(node.get("nodes"), level + 1)

    walk(tree.get("structure"), 1)
    if not heads:
        return md, 0, f"flash 未得到章节结构（toc_source={tree.get('toc_source')}），已退回无标题版本"

    parts = PAGE_MARKER_RE.split(md)  # [前导文本, 页码1, 页文本1, 页码2, 页文本2, ...]
    out, count = [parts[0]], 0
    for i in range(1, len(parts), 2):
        page, text = int(parts[i]), parts[i + 1]
        out.append(f"<!-- page: {page} -->")
        if page in heads:
            out.append("\n\n" + "\n\n".join(heads[page]) + "\n\n" + text.lstrip("\n"))
            count += len(heads[page])
        else:
            out.append(text)
    return "".join(out), count, None


SKIPPED = "skipped_no_text_layer"      # meta.json 里的 status：无文字层，已跳过
WRONG_PASSWORD = "skipped_wrong_password"  # meta.json 里的 status：需要密码但没配 / 候选密码都不对，已跳过
SKIP_LABELS = {SKIPPED: "无文字层", WRONG_PASSWORD: "密码不正确或未配置"}  # 被跳过的 PDF：状态 -> 原因（读题、汇总共用）
SKIP_STATUSES = tuple(SKIP_LABELS)


def describe_failure(exc: Exception) -> str:
    """失败原因文本；PyPDF2 读 AES 加密 PDF 缺 pycryptodome 时给出明确的中文处理办法。"""
    if "DependencyError" in type(exc).__name__ or "PyCryptodome" in str(exc):
        return ("该 PDF 是 AES 加密的，缺少依赖 pycryptodome；请运行 `uv sync --group notebook`"
                "（或 `uv add pycryptodome`）安装后重跑本 Notebook，其它文档不受影响"
                f"（原始错误：{type(exc).__name__}: {exc}）")
    return f"{type(exc).__name__}: {exc}"


def open_with_password(pdf: Path) -> tuple[bool, str | None, bool]:
    """判断 PDF 怎么打开，返回 (是否加密, 用到的密码, 能否打开)：
    - 不加密：(False, None, True)；加密但空密码即可打开（如只设了 owner 密码）：(True, None, True)——都走原有流程；
    - 需要密码：按 PDF_PASSWORDS 顺序尝试，成功 (True, 该密码, True)；没配或都不对 (True, None, False)。
    PyPDF2 的 decrypt 返回 0 = 密码不对，1 / 2 = 匹配了 user / owner 密码。"""
    import PyPDF2

    with pdf.open("rb") as fh:
        reader = PyPDF2.PdfReader(fh)  # 加密的会先自动试空密码；AES 缺 pycryptodome 在这里抛 DependencyError
        if not reader.is_encrypted:
            return False, None, True
        if int(reader.decrypt("")):
            return True, None, True
        for password in PDF_PASSWORDS:
            if int(reader.decrypt(password)):
                return True, password, True
    return True, None, False


def password_document_text(pdf: Path, password: str) -> str:
    """需要密码的 PDF：用密码打开 PyPDF2 reader 逐页抽文字层（只在内存里解密，不落解密副本）。
    输出格式与 Extractor(env={}).document_text 的文字层路径一致：每页前 `<!-- page: N -->`，页间空一行。"""
    import PyPDF2
    from superindex.extractors.azure_di import PAGE_MARKER

    with pdf.open("rb") as fh:
        reader = PyPDF2.PdfReader(fh)
        reader.decrypt(password)
        pages = [re.sub("[\ud800-\udfff]", "\ufffd", page.extract_text() or "") for page in reader.pages]  # 与引擎一样去掉孤立代理项
    parts: list[str] = []
    for number, text in enumerate(pages, start=1):
        parts += [PAGE_MARKER.format(n=number), text]
    return "\n\n".join(parts)


def decrypted_stream(pdf: Path, password: str) -> io.BytesIO:
    """给 flash 用的解密版 PDF：整份留在内存（BytesIO），不写任何文件，用完随对象释放。
    page_index_flash 底层的 pypdfium2 能传密码，但 flash 的签名没有这个参数，所以改传解密后的内容；
    writer.append 会连书签一起带过去（flash 会用到内嵌目录）。"""
    import PyPDF2

    with pdf.open("rb") as fh:
        reader = PyPDF2.PdfReader(fh)
        reader.decrypt(password)
        writer = PyPDF2.PdfWriter()
        writer.append(reader)
        buf = io.BytesIO()
        writer.write(buf)
    buf.seek(0)
    return buf


def meta_path_of(pdf: Path) -> Path:
    return (MD_DIR / pdf.relative_to(PDF_DIR)).with_suffix(".meta.json")


def write_meta(pdf: Path, **fields: Any) -> None:
    meta = {"source": str(pdf.resolve()), "extractor": PDF_EXTRACTOR, **fields,
            "extracted_at": datetime.now().isoformat(timespec="seconds")}
    path = meta_path_of(pdf)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def read_meta(pdf: Path) -> dict[str, Any]:
    try:
        return json.loads(meta_path_of(pdf).read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return {}


def extract_text_layer(pdf: Path, md_path: Path) -> dict[str, Any]:
    """一个 PDF 走文字层转 Markdown，写 .md 与 .meta.json；返回汇总表的一行。"""
    encrypted, password, openable = open_with_password(pdf)
    if not openable:  # 需要密码但没配 / 候选密码都不对：跳过并标记（reason 里不含任何密码）
        reason = ("PDF 需要密码才能打开，但没有配置 PDF_INGEST_PASSWORD / PDF_PASSWORDS" if not PDF_PASSWORDS
                  else f"PDF 需要密码才能打开，已尝试的 {len(PDF_PASSWORDS)} 个候选密码均不正确")
        write_meta(pdf, status=WRONG_PASSWORD, encrypted=True, password_used=False, pages=None, reason=reason)
        return {"status": WRONG_PASSWORD, "pages": None, "headings": 0,
                "note": f"警告：{reason}，已跳过、不建库；改好 PDF_INGEST_PASSWORD / PDF_PASSWORDS 后重跑会自动重试"}
    used = password is not None  # meta 只记"有没有用密码"，不记是哪一个
    # env={} -> 即使环境里有 Azure 密钥也绝不调用 Azure；需要密码的走内存解密版
    md = password_document_text(pdf, password) if used else Extractor(env={}).document_text(pdf)
    parts = PAGE_MARKER_RE.split(md)
    pages = [(int(parts[i]), parts[i + 1]) for i in range(1, len(parts), 2)]
    empty_pages = [n for n, text in pages if not text.strip()]
    if not pages or len(empty_pages) == len(pages):  # 所有页文字都为空 -> 无文字层，跳过并标记
        write_meta(pdf, status=SKIPPED, encrypted=encrypted, password_used=used, pages=len(pages), empty_pages=empty_pages,
                   reason="PDF 没有文字层（扫描件/纯图片），文字层抽取结果全为空；如需处理请改用 azure-di")
        return {"status": SKIPPED, "pages": len(pages), "headings": 0,
                "note": "警告：无文字层（扫描件？），已跳过、不建库；请改用 PDF_EXTRACTOR=\"azure-di\" 处理"}
    notes = [f"警告：{len(empty_pages)} 页无文字 {empty_pages}（可能是整页图片）"] if empty_pages else []
    headings = 0
    if ADD_HEADINGS:
        md, headings, warn = add_headings(md, pdf, password)
        if warn:
            notes.append("警告：" + warn)
    md_path.parent.mkdir(parents=True, exist_ok=True)
    md_path.write_text(md, encoding="utf-8")
    write_meta(pdf, status="ok", encrypted=encrypted, password_used=used, pages=len(pages), empty_pages=empty_pages,
               add_headings=ADD_HEADINGS, headings_added=bool(headings), heading_count=headings)
    return {"status": "converted", "pages": len(pages), "headings": headings, "note": "；".join(notes)}


print(f"MD_DIR {MD_DIR} 下已有 {sum(1 for _ in MD_DIR.rglob('*.md')) if MD_DIR.is_dir() else 0} 个 .md", flush=True)
pdfs = list_pdfs(PDF_DIR)
rows: list[dict[str, Any]] = []
step_started = time.perf_counter()
print(f"步骤 1：共 {len(pdfs)} 个 PDF，抽取方式 {PDF_EXTRACTOR}，补标题 {'开' if ADD_HEADINGS else '关'}", flush=True)


def progress(i: int, pdf: Path, row: dict[str, Any], seconds: float) -> None:
    """实时打印一个 PDF 的处理结果（开始处理时已先打印了 "[i/n] 文件名 ..."，这里接着写在同一行）。"""
    pages = f"，{row['pages']} 页" if row.get("pages") else ""
    heads = f"，标题 {row['headings']}" if row.get("headings") else ""
    print(f" {row['status']}{pages}{heads}（{seconds:.1f}s，累计 {time.perf_counter() - step_started:.0f}s）", flush=True)


if PDF_EXTRACTOR == "text-layer":
    for i, pdf in enumerate(pdfs, start=1):
        print(f"[{i}/{len(pdfs)}] {pdf.relative_to(PDF_DIR)} ...", end="", flush=True)
        started = time.perf_counter()
        md_path = (MD_DIR / pdf.relative_to(PDF_DIR)).with_suffix(".md")
        meta = read_meta(pdf)
        # 已标记 skipped_wrong_password 的不在这里跳过：没有 .md、状态也不是 SKIPPED，每次运行都会重新尝试打开（密码配置可能已改对）
        if not FORCE_EXTRACT and (md_path.exists() or meta.get("status") == SKIPPED):
            # 已有 .md，或已被标记为无文字层：不重复抽取（FORCE_EXTRACT=True 可强制重试）
            status = SKIPPED if meta.get("status") == SKIPPED and not md_path.exists() else "skipped_existing"
            note = ""
            if status == SKIPPED:
                note = "此前已标记为无文字层，不重复抽取；改用 azure-di 或设 FORCE_EXTRACT=True 重试"
            elif meta and (meta.get("extractor") != PDF_EXTRACTOR
                           or meta.get("add_headings") != ADD_HEADINGS):
                note = "已有 .md 的抽取方式/补标题与当前配置不同；需要按新配置重转请设 FORCE_EXTRACT=True"
            if meta.get("empty_pages") and status != SKIPPED:
                note = (note + "；" if note else "") + f"警告：{len(meta['empty_pages'])} 页无文字 {meta['empty_pages']}"
            rows.append({"pdf": pdf.name, "status": status, "pages": meta.get("pages"),
                         "headings": meta.get("heading_count"), "note": note})
            progress(i, pdf, rows[-1], time.perf_counter() - started)
            continue
        if FORCE_EXTRACT and md_path.exists():
            md_path.unlink()  # 重转：避免上一次的 .md 在新结果为"无文字层"时残留
        try:
            result = extract_text_layer(pdf, md_path)
        except Exception as exc:  # noqa: BLE001 - 单个 PDF 失败不中断其它文档
            result = {"status": "failed", "pages": None, "headings": None,
                      "note": describe_failure(exc)}
        rows.append({"pdf": pdf.name, **result})
        progress(i, pdf, rows[-1], time.perf_counter() - started)
elif PDF_EXTRACTOR == "azure-di":
    # 只有显式选了 azure-di 才会调用 Azure（需 AZURE_DI_ENDPOINT / AZURE_DI_KEY）
    from superindex.extractors.azure_di import AzureDIConfig, AzureDIError, extract_corpus

    try:
        AzureDIConfig.from_env().validate()  # 缺 AZURE_DI_ENDPOINT / AZURE_DI_KEY 时给出明确提示
    except AzureDIError as exc:
        raise RuntimeError(f"PDF_EXTRACTOR=\"azure-di\" 但 Azure 未配置：{exc}\n"
                           "或者把 PDF_EXTRACTOR 改回 \"text-layer\"（离线）。") from exc
    statuses: dict[str, str] = {}

    def on_progress(i: int, n: int, pdf: Path, status: str, info: Any) -> None:
        statuses[str(pdf)] = status
        print(f"[{i}/{n}] {Path(pdf).name} {status}（累计 {time.perf_counter() - step_started:.0f}s）", flush=True)

    summary = extract_corpus(PDF_DIR, MD_DIR, force=FORCE_EXTRACT, workers=2, page_markers=True, on_progress=on_progress)
    errors = dict(e.split(": ", 1) for e in summary["errors"])
    for pdf in pdfs:
        status = {"converted": "converted", "skipped": "skipped_existing"}.get(statuses.get(str(pdf)), "failed")
        rows.append({"pdf": pdf.name, "status": status, "pages": read_meta(pdf).get("pages"), "headings": None,
                     "note": errors.get(pdf.name, "")})
else:
    raise ValueError(f"PDF_EXTRACTOR 只能是 \"text-layer\" 或 \"azure-di\"，当前是 {PDF_EXTRACTOR!r}")

# 汇总所有被跳过的 PDF（含此前运行标记的），写到 WORK_DIR/skipped_pdfs.json
skipped_pdfs = [{"pdf": str(p.relative_to(PDF_DIR)), "source": str(p.resolve()), "status": read_meta(p)["status"],
                 "reason": read_meta(p).get("reason"), "pages": read_meta(p).get("pages"),
                 "marked_at": read_meta(p).get("extracted_at")}
                for p in pdfs if read_meta(p).get("status") in SKIP_STATUSES]
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "skipped_pdfs.json").write_text(json.dumps(skipped_pdfs, ensure_ascii=False, indent=2), encoding="utf-8")

_counts: dict[str, int] = {}
for row in rows:
    _counts[row["status"]] = _counts.get(row["status"], 0) + 1
print(f"步骤 1 完成：{len(rows)} 个 PDF，用时 {time.perf_counter() - step_started:.0f}s；"
      + "，".join(f"{k} {v}" for k, v in sorted(_counts.items())), flush=True)
display(pd.DataFrame(rows))
for row in rows:
    if row["status"] in (*SKIP_STATUSES, "failed") or "警告" in row["note"]:
        print(f"[{row['status']}] {row['pdf']}: {row['note']}")
print(f"被跳过的 PDF：{len(skipped_pdfs)} 个，清单见 {WORK_DIR / 'skipped_pdfs.json'}")

## 步骤 2：建库

对 `md/` 下每个 Markdown 调 `index_markdown`（`pdf=` 记录源 PDF 关联）。内容没变会自动跳过；
`INDEX_SUMMARY=False` 时不调 LLM。

In [ ]:
settings = LLMSettings.resolve()
summary_model = None
if INDEX_SUMMARY:
    summary_model = settings.require("index")  # 没配 SUPERINDEX_INDEX_MODEL 会给出明确提示
    configure_litellm()

md_files = sorted(MD_DIR.rglob("*.md")) if MD_DIR.is_dir() else []
if not md_files:
    raise RuntimeError(f"{MD_DIR} 下没有 Markdown，请先成功运行步骤 1")
pdf_by_stem = page_images.pdf_index(PDF_DIR)

index_rows: list[dict[str, Any]] = []
# index_markdown 内部会 asyncio.run(...)，而 Notebook 的主线程已有运行中的事件循环，直接调用会报
# "asyncio.run() cannot be called from a running event loop"；放到单独的工作线程里执行（该线程没有事件循环）。
index_pool = ThreadPoolExecutor(max_workers=1)


def pdf_to_link(md_path: Path) -> Path | None:
    """index_markdown(pdf=...) 的源 PDF 关联，只为页面截图服务（默认关闭）。
    步骤 1 用密码打开的 PDF 不关联：页面截图用的 pypdfium2 没有密码就打不开，传 pdf= 只会得到一条误导性警告
    （"PDF ... not linked"）——即需要密码的 PDF 不支持页面截图，文字问答不受影响。"""
    try:
        if json.loads(md_path.with_suffix(".meta.json").read_text(encoding="utf-8")).get("password_used"):
            return None
    except (OSError, ValueError):
        pass
    return page_images.find_pdf(md_path, pdf_by_stem)


index_started = time.perf_counter()
print(f"步骤 2：共 {len(md_files)} 个文档，摘要 {'开（调用模型）' if INDEX_SUMMARY else '关'}", flush=True)
for i, md_path in enumerate(md_files, start=1):
    print(f"[{i}/{len(md_files)}] {md_path.relative_to(MD_DIR)} ...", end="", flush=True)
    started = time.perf_counter()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = index_pool.submit(index_markdown, md_path, STORE, summary_model=summary_model,
                                    backend=settings.index_backend(), force=False,
                                    pdf=pdf_to_link(md_path)).result()
        index_rows.append({"doc_id": res.doc_id, "文档": res.name, "页数": res.pages, "节点数": res.nodes,
                           "状态": "跳过（未变）" if res.skipped else "已建库",
                           "页码标记": res.has_markers, "警告": "；".join(res.warnings)})
    except Exception as exc:  # noqa: BLE001 - 单个文档失败不中断其它文档
        index_rows.append({"doc_id": None, "文档": md_path.name, "页数": None, "节点数": None,
                           "状态": f"失败：{type(exc).__name__}: {exc}", "页码标记": None, "警告": ""})
    print(f" {index_rows[-1]['状态'][:60]}（{time.perf_counter() - started:.1f}s，"
          f"累计 {time.perf_counter() - index_started:.0f}s）", flush=True)
index_pool.shutdown()
print(f"步骤 2 完成：{len(index_rows)} 个文档，用时 {time.perf_counter() - index_started:.0f}s", flush=True)
display(pd.DataFrame(index_rows))
sync_to_persist("建库后")

## 步骤 3：读问题

按 `FIELD_MAP` 把数据集规范化成 `batch.Question`（没有 id 时自动编号 `Q001…`；其余字段进 `extra`；映射值支持 `a.b`、`evidence[].file` 这类路径，见配置 cell），
再检查每题的 `doc` 能否在库里匹配到（复用 `batch._scope`，含用 PDF 文件名 stem 匹配）。匹配不到的题仍会运行，在结果里记为错误。汇总里还会给出按 `doc` 范围的题数分布（全库 / 单文档 / 多文档）。

In [ ]:
# results.jsonl 自己占用的字段名；数据集里与之同名的额外字段会被加前缀 src_，避免覆盖结果字段
STANDARD_KEYS = {"id", "question", "doc", "expected", "scope", "answer", "error", "seconds", "llm_turns",
                 "tool_calls", "pages_read", "score", "prefetch", "prefetch_hit", "page_images", "image_count", "status"}


def load_records(path: Path, records_key: str | None) -> list[Any]:
    """读 .json（顶层列表，或 {"questions": [...]} / {RECORDS_KEY: [...]}）或 .jsonl。"""
    if not path.is_file():
        raise FileNotFoundError(f"问题文件不存在：{path.resolve()}（请修改配置里的 DATASET_PATH）")
    text = path.read_text(encoding="utf-8-sig")
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix != ".json":
        raise ValueError(f"只支持 .json / .jsonl，当前是 {path.suffix!r}")
    data = json.loads(text)
    if isinstance(data, list):
        return data
    if not isinstance(data, dict):
        raise TypeError(f"{path.name}: 顶层应是列表或对象")
    key = records_key or ("questions" if "questions" in data else None)
    if key is None or not isinstance(data.get(key), list):
        candidates = [k for k, v in data.items() if isinstance(v, list)]
        raise ValueError(f"{path.name}: 顶层是对象，找不到题目列表；请设置 RECORDS_KEY，候选键：{candidates}")
    return data[key]


def _walk(node: Any, segments: list[str]) -> Any:
    """沿路径段取值；取不到返回 None。段以 "[]" 结尾表示展开列表，对每个元素继续取后续路径。"""
    if not segments:
        return node
    segment, rest = segments[0], segments[1:]
    spread = segment.endswith("[]")
    key = segment[:-2] if spread else segment
    if key:
        if not isinstance(node, dict) or key not in node:
            return None
        node = node[key]
    if not spread:
        return _walk(node, rest)
    if not isinstance(node, list):
        return None
    out: list[Any] = []
    for element in node:
        value = _walk(element, rest)
        out.extend(value if isinstance(value, list) else [value])
    return out


def resolve_path(item: dict[str, Any], spec: str) -> Any:
    """按 spec 取值：顶层字段名、"a.b" 嵌套路径、"a[].b" 列表展开；取不到返回 None。
    展开得到的列表会去重（保持首次出现顺序）并丢弃空值。"""
    if spec in item:  # 顶层字段名直接命中（含名字里本身带 "." 的字段）
        return item[spec]
    value = _walk(item, spec.split("."))
    if isinstance(value, list) and "[]" in spec:
        seen: dict[str, Any] = {}
        for v in value:
            if v is not None and v != "":
                seen.setdefault(json.dumps(v, ensure_ascii=False, sort_keys=True), v)
        value = list(seen.values())
    return value


def scalar_of(value: Any) -> Any:
    """id / question / expected 取到列表：空 -> 缺省，1 个元素取该元素，多个用换行拼接。"""
    if isinstance(value, list):
        return None if not value else value[0] if len(value) == 1 else "\n".join(str(v) for v in value)
    return value


def to_questions(items: list[Any], field_map: dict[str, str | None]) -> list[Question]:
    """按字段映射规范化；字段名为 None 或数据里没有该字段（含路径取不到）都当作缺省。"""
    names = {k: field_map.get(k) for k in ("id", "question", "expected", "doc", "pages")}
    # 只有"纯顶层字段名"的映射才把源字段从 extra 里拿走；路径映射（如 evidence[].file）保留整个源字段
    mapped = {v for v in names.values() if v and not re.search(r"[.\[\]]", v)}
    questions: list[Question] = []
    for index, item in enumerate(items, start=1):
        if isinstance(item, str) and names["question"]:
            item = {names["question"]: item}
        if not isinstance(item, dict):
            raise TypeError(f"第 {index} 条：应是对象，实际是 {item!r}")
        values = {k: resolve_path(item, v) for k, v in names.items() if v and k != "pages"}
        for key in ("id", "question", "expected"):
            if key in values:
                values[key] = scalar_of(values[key])
        q = _from_item(values, index)  # 复用 batch 的规范化与自动编号（doc 为列表即多文档）
        if q is None:
            print(f"  跳过第 {index} 条：问题为空")
            continue
        q.extra = {(f"src_{k}" if k in STANDARD_KEYS else k): v  # 其余字段原样带进结果
                   for k, v in item.items() if k not in mapped}
        if names["pages"]:  # batch.gold_pages 读 extra["pages"]：列表，元素为 12 或 "12-13"；多文档为页码并集
            pages = resolve_path(item, names["pages"])
            pages = pages if isinstance(pages, list) else [] if pages in (None, "") else [pages]
            if pages:
                q.extra["pages"] = pages
        questions.append(q)
    seen: set[str] = set()
    for q in questions:
        if q.id in seen:
            raise ValueError(f"问题 id 重复：{q.id!r}")
        seen.add(q.id)
    return questions


items = load_records(DATASET_PATH, RECORDS_KEY)
questions = to_questions(items, FIELD_MAP)
if not questions:
    first = items[0] if items else None
    raise ValueError(f"没有读到任何问题；请检查 FIELD_MAP[\"question\"]。数据第一条：{first!r}")
if LIMIT:
    questions = questions[:LIMIT]

store_docs = [m for m in DocStore(str(STORE)).list_metas() if m.get("status") == "completed"]
# 被步骤 1 跳过的 PDF（无文字层 / 密码不正确或未配置；见 meta.json、skipped_pdfs.json）
skipped_names = {Path(p["pdf"]).name: p for p in skipped_pdfs}


def skipped_pdf_of(q: Question) -> list[str]:
    """q.doc 的每一项都在库里匹配不到、且至少一项对应被跳过的 PDF 时，返回这些 PDF 的文件名。"""
    hit: list[str] = []
    for want in q.doc:
        try:
            _scope(store_docs, [want])
            return []  # 有项能在库里匹配到：照常提问
        except ConfigError:
            hit += [n for n in skipped_names if Path(want).stem.lower() in Path(n).stem.lower()]
    return hit


skipped_q: dict[str, tuple[str, str]] = {}  # 题 id -> (状态, 跳过原因)（这些题不会去问模型）
unmatched: list[tuple[str, list[str], str]] = []
for q in questions:
    if q.doc:
        if names := skipped_pdf_of(q):
            status = skipped_names[names[0]]["status"]  # 指向多个被跳过的 PDF 且原因不同时，以第一个为准
            names = [n for n in names if skipped_names[n]["status"] == status]
            skipped_q[q.id] = (status, f"跳过：源 PDF {SKIP_LABELS[status]}（{', '.join(dict.fromkeys(names))}）")
            continue
        try:
            _scope(store_docs, q.doc)
        except ConfigError as exc:
            unmatched.append((q.id, q.doc, str(exc)))

print(f"题数 {len(questions)}（库内文档 {len(store_docs)} 个）；"
      f"{sum(1 for q in questions if q.doc)} 题指定了文档，{sum(1 for q in questions if not q.doc)} 题在全库范围提问")
print(f"doc 范围分布：全库 {sum(1 for q in questions if not q.doc)} 题 / "
      f"单文档 {sum(1 for q in questions if len(q.doc) == 1)} 题 / "
      f"多文档 {sum(1 for q in questions if len(q.doc) > 1)} 题；"
      f"带 pages 的 {sum(1 for q in questions if q.extra.get('pages'))} 题（按页码判定检索命中）")
for status, label in SKIP_LABELS.items():
    group = {qid: why for qid, (st, why) in skipped_q.items() if st == status}
    if group:
        print(f"以下 {len(group)} 题指向{label}的 PDF，不会去问模型，直接在结果里记为 {status}：")
        for qid, why in group.items():
            print(f"  {qid}: {why}")
if unmatched:
    print(f"以下 {len(unmatched)} 题的 doc 在库里匹配不到（运行时会记为错误，请检查文档名或 PDF 是否已转换）：")
    for qid, doc, why in unmatched:
        print(f"  {qid}: {doc}")
display(pd.DataFrame([{"id": q.id, "question": q.question, "expected": q.expected,
                       "doc": "; ".join(q.doc), **q.extra} for q in questions]).head(10))

## 步骤 4：逐题回答

流程与 `superindex batch`（`cmd_batch`）对齐：同一个 `make_client`（含 `search_pages` / `calculate` 工具）、关键词检索前置（prefetch）、`run_question`，
记录字段与 `results.jsonl` 完全兼容，每完成一题立即追加写入；单题失败（含超时）只记在该题的 `error` 里，不中断整批。
`RESUME=True` 时续跑最近一次运行目录：跳过已成功的题，重跑出错的题。

In [ ]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
settings = LLMSettings.resolve()
instructions = resolve_instructions()

existing_runs = sorted(p for p in RUNS_DIR.glob("*") if (p / RESULTS_FILE).is_file())
run_dir = existing_runs[-1] if RESUME and existing_runs else RUNS_DIR / datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)
results_path = run_dir / RESULTS_FILE
results_path.touch(exist_ok=True)

done = {k: r for k, r in read_results(run_dir).items() if not r.get("error")} if RESUME else {}
todo = [q for q in questions if q.id not in done]
prefetch_k = prefetch.resolve_k(PREFETCH, PREFETCH_K)
image_mode = page_images.resolve_mode(None)
started = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"运行目录 : {run_dir}")
print(f"共 {len(questions)} 题：已完成 {len(questions) - len(todo)}，本次运行 {len(todo)}；"
      f"timeout {TIMEOUT:g}s，并发 {CONCURRENCY}，prefetch {prefetch_k or '关'}，page image {image_mode}")

def with_extra_headers(client: Any) -> Any:
    """让 client.chat 的每次调用都带上 LLM_EXTRA_HEADERS（run_question / image_chat.chat 不透传 extra_headers）。
    每次调用时才读取 LLM_EXTRA_HEADERS，中途改了字典（如刷新 token）下一题就生效。只覆盖问答路径。"""
    chat = client.chat

    def chat_with_headers(*args: Any, **kwargs: Any) -> Any:
        kwargs["extra_headers"] = {**LLM_EXTRA_HEADERS, **(kwargs.get("extra_headers") or {})}
        return chat(*args, **kwargs)

    client.chat = chat_with_headers
    return client


client = None
if any(q.id not in skipped_q for q in todo):  # 只剩被跳过的题时不需要模型
    try:
        client = make_client(settings, STORE, instructions=instructions)  # 没配 SUPERINDEX_CHAT_MODEL 会在这里报错
        if LLM_EXTRA_HEADERS:
            client = with_extra_headers(client)
    except ConfigError as exc:
        raise RuntimeError(f"未配置可用的问答模型：{exc}\n"
                           "请设置 SUPERINDEX_CHAT_MODEL，或设置 OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY"
                           "（.env 或「模型配置」cell 的 LLM_CONFIG）") from exc

lock = threading.Lock()
page_texts: dict[str, list[str]] = {}
bar = tqdm(total=len(todo), desc="回答问题", unit="题") if tqdm and todo else None


def say(text: str) -> None:
    (bar.write if bar else print)(text)


class _SkippedQuestion(Exception):
    """指向被跳过 PDF（无文字层 / 密码不正确或未配置）的题：不问模型。"""


def answer_one(q: Question) -> None:
    try:
        if q.id in skipped_q:  # 指向被跳过的 PDF：不问模型，只记一条说明
            raise _SkippedQuestion(skipped_q[q.id][1])
        scope_ids = _scope(store_docs, q.doc) if q.doc else None
        scope = scope_ids[0] if scope_ids and len(scope_ids) == 1 else scope_ids
        hits = prefetch.search(STORE, q.question, scope_ids, prefetch_k)
        session = page_images.new_session(STORE, image_mode)
        if session is not None:
            session.attach_prefetch(hits)
        record = run_question(client, q, scope, timeout=TIMEOUT, reasoning_effort=settings.reasoning_effort,
                              message=prefetch.augment(q.question, hits), session=session)
        if prefetch_k:
            judge, flags = judge_hits(STORE, q, hits, page_texts)
            record["prefetch"] = [{"doc_name": h.doc_name, "page": h.page, "relevant": f}
                                  for h, f in zip(hits, flags)]
            record["prefetch_hit"] = any(flags) if judge else None
    except Exception as exc:  # noqa: BLE001 - 如 doc 匹配不到：记下来，整批继续
        skipped = isinstance(exc, _SkippedQuestion)
        record = {"id": q.id, "question": q.question, "doc": q.doc, "expected": q.expected, **q.extra,
                  "scope": None, "answer": "", "error": str(exc) if skipped else f"{type(exc).__name__}: {exc}",
                  "seconds": 0.0, "llm_turns": 0, "tool_calls": [], "pages_read": [],
                  "score": score(q.expected, "")}
        if skipped:
            record["status"] = skipped_q[q.id][0]
    with lock:  # 每题完成立即追加写入
        with results_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(record, ensure_ascii=False) + "\n")
        if bar:
            bar.update(1)
        status = (record["error"] if record.get("status") in SKIP_STATUSES else "ERROR " + record["error"]) if record["error"] else \
            ("命中" if (record["score"] or {}).get("hit") else ("未命中" if record["score"] else "已回答"))
        say(f"{q.id}  {record['seconds']:.1f}s  {status}  读取页: {', '.join(record['pages_read']) or '-'}")


t0 = time.time()
if todo:
    with ThreadPoolExecutor(max_workers=max(1, CONCURRENCY)) as pool:
        list(pool.map(answer_one, todo))
if bar:
    bar.close()
wall = time.time() - t0
print(f"本次运行 {len(todo)} 题，用时 {wall:.1f}s")

## 步骤 5：整理保存

`read_results` + `write_summary` 生成 `summary.md`；再用 pandas 导出 `answers.csv`（utf-8-sig，Excel 直接打开不乱码）和 `answers.xlsx`。

In [ ]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

latest = read_results(run_dir)
records = [latest[q.id] for q in questions if q.id in latest]
write_summary(records, run_dir / SUMMARY_FILE, {
    "started": started, "questions": str(DATASET_PATH), "store": str(STORE),
    "chat_model": settings.chat_model, "wall_seconds": wall, "prefetch_k": prefetch_k, "page_image": image_mode})

def flat(value: Any) -> Any:
    return json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value


def to_row(r: dict[str, Any]) -> dict[str, Any]:
    sc = r.get("score")
    row = {"id": r.get("id"), "question": r.get("question"), "answer": r.get("answer"),
           "expected": r.get("expected"), "score": ("跳过" if r.get("status") in SKIP_STATUSES else "出错" if r.get("error") else ("命中" if sc["hit"] else "未命中") if sc else None),
           "doc": "; ".join(r.get("doc") or []), "pages_read": ", ".join(r.get("pages_read") or []),
           "seconds": r.get("seconds"), "llm_turns": r.get("llm_turns"), "error": r.get("error"),
           "status": r.get("status") or ("error" if r.get("error") else "ok")}
    row.update({k: flat(v) for k, v in r.items() if k not in STANDARD_KEYS})  # 数据集带来的额外字段
    return row


df = pd.DataFrame([to_row(r) for r in records])
for col in ("expected", "score"):  # 数据集没有 expected 时不输出这两列
    if col in df and df[col].isna().all():
        df = df.drop(columns=col)

csv_path, xlsx_path = run_dir / "answers.csv", run_dir / "answers.xlsx"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
def clean_cell(v: Any) -> Any:
    return ILLEGAL_CHARACTERS_RE.sub("", v)[:32767] if isinstance(v, str) else v  # xlsx 不接受控制字符，单元格最长 32767 字符


# DataFrame.map 需要 pandas>=2.1；更旧的版本退回 applymap（这样不必升级集群预装的 pandas）
(df.map(clean_cell) if hasattr(df, "map") else df.applymap(clean_cell)).to_excel(xlsx_path, index=False)

skip_counts = {s: sum(1 for r in records if r.get("status") == s) for s in SKIP_STATUSES}
skipped = sum(skip_counts.values())
errors = sum(1 for r in records if r.get("error")) - skipped
scored = [r for r in records if r.get("score") and not r.get("error")]
hits = sum(1 for r in scored if r["score"]["hit"])
print(f"题数 {len(records)}：成功 {len(records) - errors - skipped}，失败 {errors}，跳过 {skipped}"
      + (f"（{'，'.join(f'{SKIP_LABELS[s]} {n}' for s, n in skip_counts.items() if n)}）" if skipped else "")
      + (f"；粗评分命中 {hits}/{len(scored)}（{hits / len(scored):.0%}，仅供筛查，需人工复核）" if scored else ""))
print("产出文件：")
for path in (run_dir / RESULTS_FILE, run_dir / SUMMARY_FILE, csv_path, xlsx_path):
    print(f"  {path}")
display(df)
sync_to_persist("全部完成")

## 步骤 6：逐题打印答案

结果已落盘后，把 `records`（步骤 5 读出的完整记录，按题集原顺序，含续跑前已成功的题）里每个问题的 id 和答案一个一个打印出来；答案不截断，出错的题打印 status 和 error。

In [ ]:
total = len(records)
for n, r in enumerate(records, start=1):
    print("=" * 80, flush=True)
    print(f"[{n}/{total}] id: {r.get('id')}", flush=True)
    print(f"问题: {r.get('question')}", flush=True)
    if r.get("error"):
        print(f"状态: {r.get('status') or 'error'}\n错误: {r['error']}", flush=True)
    print(f"答案:\n{r.get('answer') or '（无答案）'}", flush=True)
print("=" * 80, flush=True)

## 步骤 7：答案评测（deepeval，可选）

用 deepeval 的 `GEval` 指标（`答案正确性`，LLM-as-judge）让裁判模型对照 **期望答案** 逐题打分（0~1，`≥ PASS_THRESHOLD` 判为通过），输出逐题结果与准确率。

- **只评答案正确性**（问题 + 期望答案 + 实际回答），不做 Faithfulness：`results.jsonl` 里只有读取过的页码，没有检索到的页面原文。
- 没有 `expected` 的题、答题没成功的题（出错、`skipped_no_text_layer` 等）不送裁判，标为 `not_evaluated` 并写明原因。
- **断点续跑**：`eval_results.jsonl` 里已有结论（且问题、期望答案、回答都没变）的题直接跳过；每评完一题立刻追加写盘。评测出错（`error`）的题下次重跑。
- **输入**（打印最终用的绝对路径）：`RESULTS_PATH`（`results.jsonl` 或它所在的运行目录；相对路径以 `ROOT_DIR` 为基准）> 本次运行的 `run_dir/results.jsonl`（步骤 4 的运行目录）> `RUNS_DIR` 下最新时间戳目录的 `results.jsonl`。
  **输出**固定写在 `ROOT_DIR/data/eval/<问题集文件名>/` 下（不可配置）：`eval_results.jsonl`、`eval_summary.md`、`eval.csv`、`eval.xlsx`，以及 deepeval 的缓存目录 `.deepeval`；评测对 `WORK_DIR` / 运行目录只读不写，也不同步到 `PERSIST_DIR`。输出目录若落在 `WORK_DIR` 或 `PERSIST_DIR` 之内（如把 `NB_WORK_DIR` 设成了 `data`）会跳过并说明原因。
- **裁判需要 OpenAI 兼容端点**（Chat Completions，`openai` 包）：用「模型配置」/ `.env` 的 `OPENAI_MODEL` / `OPENAI_BASE_URL` / `OPENAI_API_KEY`（没设 `OPENAI_MODEL` 时回退到 `openai/` 前缀的 `SUPERINDEX_CHAT_MODEL` 及 `SUPERINDEX_BASE_URL` / `SUPERINDEX_API_KEY_OVERRIDE`），带上 `LLM_EXTRA_HEADERS`；
  想用另一个（更强的）模型当裁判设 `JUDGE_MODEL`。问答走公司 SDK（`USE_CUSTOM_LLM`）、Databricks Model Serving、Azure 等非 OpenAI 兼容配置时，裁判拿不到端点，本步骤跳过。
- **可选**：以下情况只打印一句提示、跳过本步骤（不报错，Run All 照样跑完）：`RUN_EVAL=False`；没装 deepeval（Databricks 上运行「环境安装」里的 `%pip install deepeval` cell；本地用 `uv run --group notebook --with deepeval ...` 启动，deepeval 不进项目依赖）；找不到 `results.jsonl`；输出目录落在 `WORK_DIR` / `PERSIST_DIR` 之内；没有带 `expected` 且答题成功的题；没配裁判模型。

配置（在下面 cell 顶部，都可用同名环境变量覆盖，也可写进 `ROOT_DIR/.env`）：

| 配置项 | 默认 | 含义 |
|---|---|---|
| `RUN_EVAL` | `True` | 总开关 |
| `RESULTS_PATH` | 空 | 指定要评测的 `results.jsonl`（或运行目录）；空 = 按上面的顺序自动定位 |
| `JUDGE_MODEL` | 空 | 裁判模型名；空 = 用 `OPENAI_MODEL`（或 `SUPERINDEX_CHAT_MODEL` 去掉 `openai/` 前缀） |
| `PASS_THRESHOLD` | `0.5` | GEval 得分 ≥ 它判为通过 |
| `JUDGE_CONCURRENCY` | `4` | 同时评测的题数 |
| `JUDGE_TIMEOUT` | `120` | 单次裁判请求超时（秒） |
| `JUDGE_TEMPERATURE` | `0` | 模型不接受 temperature 时设 `none`（不传） |
| `JUDGE_FAKE` | `False` | 仅供自测：不联网的假裁判，走通整条 deepeval 链路 |

裁判是 deepeval 的 `DeepEvalBaseLLM` 子类 `OpenAIJudge`：deepeval 要结构化输出时先带 `response_format={"type": "json_object"}`（端点不支持就去掉重试），按 schema 解析，失败就截取文本里的 `{...}` 再解析，还不行交给 deepeval 自己解析。判定标准写在 `EVALUATION_STEPS` 里，可按需修改；准确率 = 通过数 / 已评测数（不含未评测和报错的题）。

In [ ]:
# ───────────── 步骤 7 配置与裁判定义（每项都可用同名环境变量覆盖，也可写进 ROOT_DIR/.env）─────────────
RUN_EVAL = _env("RUN_EVAL", True, _bool)             # 总开关：False 跳过整个步骤 7
# 要评测的 results.jsonl（也可以填它所在的运行目录）；空 = 本次运行的 run_dir，没有就取 RUNS_DIR 下最新的一次。
# Databricks 上如 "/Volumes/<catalog>/<schema>/<volume>/superindex_nb/<题集名>/runs/<时间戳>/results.jsonl"
RESULTS_PATH = _env("RESULTS_PATH", "")
JUDGE_MODEL = _env("JUDGE_MODEL", "")                # 裁判模型名；空 = 用 OPENAI_MODEL（或 SUPERINDEX_CHAT_MODEL 去掉 openai/ 前缀）
PASS_THRESHOLD = _env("PASS_THRESHOLD", 0.5, float)  # GEval 得分（0~1）≥ 它判为通过
JUDGE_CONCURRENCY = _env("JUDGE_CONCURRENCY", 4, int)  # 同时评测的题数
JUDGE_TIMEOUT = _env("JUDGE_TIMEOUT", 120.0, float)    # 单次裁判请求超时（秒）
JUDGE_TEMPERATURE: float | None = _env("JUDGE_TEMPERATURE", 0.0,  # 模型不接受 temperature 时设 none（不传）
                                       lambda t: None if t.lower() in ("none", "null", "off") else float(t))
# 仅供自测：JUDGE_FAKE=1 时用不联网的假裁判（按"期望答案里的数字 / 文字是否出现在回答里"打分），走通整条 deepeval 链路
JUDGE_FAKE = _env("JUDGE_FAKE", False, _bool)


def locate_results() -> Path | None:
    """RESULTS_PATH > 本次运行的 run_dir > RUNS_DIR 下最新时间戳目录；找不到返回 None。"""
    if RESULTS_PATH:
        path = under_root(RESULTS_PATH)
        return path / RESULTS_FILE if path.is_dir() else path
    if "run_dir" in globals() and (run_dir / RESULTS_FILE).is_file():  # 步骤 4 的运行目录（还在内存里）
        return (run_dir / RESULTS_FILE).resolve()
    runs = sorted(RUNS_DIR.glob(f"*/{RESULTS_FILE}"))  # 目录名是时间戳，按名字排序即按时间
    return runs[-1].resolve() if runs else None


def judge_settings() -> tuple[str, str, str]:
    """(模型名, base_url, api_key)：OPENAI_* 优先；没设 OPENAI_MODEL 时用 openai/ 前缀的 SUPERINDEX_*。"""
    model = os.environ.get("OPENAI_MODEL", "").strip()
    base_url = os.environ.get("OPENAI_BASE_URL", "").strip()
    api_key = os.environ.get("OPENAI_API_KEY", "").strip()
    chat_model = os.environ.get("SUPERINDEX_CHAT_MODEL", "").strip()
    if not model and chat_model.startswith("openai/"):
        model = chat_model.removeprefix("openai/")
        base_url = os.environ.get("SUPERINDEX_BASE_URL", "").strip() or base_url
        api_key = os.environ.get("SUPERINDEX_API_KEY_OVERRIDE", "").strip() or api_key
    return JUDGE_MODEL or model, base_url, api_key


def read_jsonl(path: Path) -> dict[str, dict[str, Any]]:
    """每个 id 的最后一条记录（保持首次出现的顺序；与 batch.read_results 一致）。"""
    out: dict[str, dict[str, Any]] = {}
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                out[str(rec.get("id"))] = rec
    return out


def eval_skip_reason(r: dict[str, Any]) -> str | None:
    """不送裁判的原因；None = 要评测。"""
    if r.get("status"):
        return f"batch_qa 未回答：{r['status']}" + (f"（{r['error']}）" if r.get("error") else "")
    if r.get("error"):
        return f"batch_qa 出错：{r['error']}"
    if r.get("expected") in (None, ""):
        return "没有期望答案（expected）"
    return None


# 评测输出固定写到 ROOT_DIR/data/eval/<题集名>/（只读 results.jsonl，不往 WORK_DIR / run 目录里写任何东西）
EVAL_DIR = ROOT_DIR / "data" / "eval" / DATASET_PATH.stem
EVAL_RESULTS = EVAL_DIR / "eval_results.jsonl"
EVAL_SUMMARY = EVAL_DIR / "eval_summary.md"
EVAL_CSV = EVAL_DIR / "eval.csv"
EVAL_XLSX = EVAL_DIR / "eval.xlsx"


def eval_dir_clash() -> str | None:
    """输出目录落在 WORK_DIR / PERSIST_DIR 之内（会污染工作目录或被同步走），或落在只读的 PDF_DIR 之内时，返回原因。"""
    for name, base in (("WORK_DIR", WORK_DIR), ("PERSIST_DIR", PERSIST_DIR), ("PDF_DIR", PDF_DIR)):
        if base is not None and EVAL_DIR.resolve().is_relative_to(base.resolve()):
            return (f"评测输出目录 {EVAL_DIR} 落在 {name}（{base}）之内；评测输出必须与 WORK_DIR / PERSIST_DIR 分开、且不能写进只读的 PDF_DIR，"
                    f"请检查 NB_WORK_DIR / PERSIST_DIR / PDF_DIR 的设置")
    return None


EVAL_SKIP_REASON: str | None = None  # 非 None = 跳过步骤 7（后面两个 cell 也安静跳过）
INPUT_PATH = locate_results() if RUN_EVAL else None
if not RUN_EVAL:
    EVAL_SKIP_REASON = "RUN_EVAL=False（要评测就设 RUN_EVAL=True，或环境变量 RUN_EVAL=1）"
elif eval_dir_clash():
    EVAL_SKIP_REASON = eval_dir_clash()
elif INPUT_PATH is None or not INPUT_PATH.is_file():
    EVAL_SKIP_REASON = (f"RESULTS_PATH 指向的 {INPUT_PATH} 不存在" if RESULTS_PATH else f"{RUNS_DIR} 下没有 {RESULTS_FILE}") + \
        "；先跑完步骤 4，或设 RESULTS_PATH 指向 results.jsonl（或它所在的运行目录）"
else:
    # 必须在 import deepeval 之前设：关闭匿名遥测与版本更新提示；不让它自己读 cwd 下的 .env（已从 ROOT_DIR/.env 读过）；
    # 本地状态目录默认是 cwd 下的 .deepeval，固定到 EVAL_DIR 里
    os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "1")
    os.environ.setdefault("DEEPEVAL_UPDATE_WARNING_OPT_IN", "0")
    os.environ.setdefault("DEEPEVAL_DISABLE_DOTENV", "1")
    os.environ.setdefault("DEEPEVAL_CACHE_FOLDER", str(EVAL_DIR / ".deepeval"))
    try:
        from deepeval.metrics import GEval
        from deepeval.models import DeepEvalBaseLLM
        from deepeval.test_case import LLMTestCase

        try:
            from deepeval.test_case import SingleTurnParams as _Params  # deepeval 4.x 的新名字
        except ImportError:  # 旧版 deepeval
            from deepeval.test_case import LLMTestCaseParams as _Params
        import openai
        from pydantic import BaseModel, ValidationError
    except ImportError as exc:
        EVAL_SKIP_REASON = (f"没有安装 deepeval（{exc}）。Databricks 上运行顶部「环境安装」里的 `%pip install deepeval` cell"
                            "（随后重启 Python）；本地用 `uv run --group notebook --with deepeval jupyter ...` 启动（deepeval 不进项目依赖）")

if EVAL_SKIP_REASON is None:
    eval_inputs = read_jsonl(INPUT_PATH)
    eval_previous = read_jsonl(EVAL_RESULTS)
    eval_done = {k for k, e in eval_previous.items() if e.get("eval_status") in ("evaluated", "not_evaluated")
                 and k in eval_inputs and all(e.get(f) == eval_inputs[k].get(f) for f in ("question", "expected", "answer"))}
    eval_todo = [r for k, r in eval_inputs.items() if k not in eval_done]
    eval_n_skip = sum(1 for r in eval_todo if eval_skip_reason(r))
    _judge_model, _judge_base_url = judge_settings()[:2]  # 不取 key，避免留在变量里被打印
    if not any(eval_skip_reason(r) is None for r in eval_inputs.values()):
        EVAL_SKIP_REASON = f"{INPUT_PATH} 里没有可评测的题（需要带 expected 且答题成功）"
    elif len(eval_todo) > eval_n_skip and not JUDGE_FAKE and not _judge_model:
        EVAL_SKIP_REASON = ("未配置裁判模型（裁判需要 OpenAI 兼容端点）：请设置 OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY"
                            "（.env 或「模型配置」cell 的 LLM_CONFIG），或配置项 / 环境变量 JUDGE_MODEL"
                            + (f"；当前问答模型 {os.environ['SUPERINDEX_CHAT_MODEL']} 不是 openai/ 前缀，裁判用不了"
                               if os.environ.get("SUPERINDEX_CHAT_MODEL", "").strip() else ""))

if EVAL_SKIP_REASON is not None:
    print(f"步骤 7 跳过：{EVAL_SKIP_REASON}")
else:
    EVALUATION_STEPS = [
        "以期望答案为准，找出实际回答给出的最终结论（最终数值或最终表述），判断它与期望答案是否一致。",
        "数值允许单位换算与合理四舍五入：如 18.1 亿 = 1,810,000,000 = 1.81B，38% = 0.38；只要数值等价就算正确。",
        "实际回答里出现了正确数字、但最终结论是另一个数，或前后自相矛盾、给出多个互相冲突的答案，算错误。",
        "实际回答额外多说的内容（解释、出处、计算过程）不扣分，只要最终结论正确就算正确。",
        "实际回答明确表示无法找到、无法回答或信息不足，而期望答案有具体内容时，算错误。",
        "期望答案是文字（非数值）时，按语义是否一致判断，不要求字面一致。",
        "完全正确给 10 分，结论错误给 0 分；只有在部分正确（如多项答案只答对一部分）时才给中间分。理由用中文简要说明。",
    ]

    def _json_block(text: str) -> str | None:
        start, end = text.find("{"), text.rfind("}")
        return text[start:end + 1] if start != -1 and end > start else None

    def _parse(text: str, schema: type[BaseModel] | None) -> Any:
        """按 schema 解析；失败就截取文本里的 {...} 再解析；还不行返回原文（deepeval 会自己再尝试解析）。"""
        if schema is None:
            return text
        for candidate in (text, _json_block(text)):
            if candidate:
                try:
                    return schema.model_validate_json(candidate)
                except (ValidationError, ValueError):
                    pass
        return text

    class OpenAIJudge(DeepEvalBaseLLM):
        """OpenAI 兼容 Chat Completions 端点上的裁判模型。"""

        def __init__(self, model: str, base_url: str | None, api_key: str | None):
            self.model_name, self.base_url, self.api_key = model, base_url or None, api_key or "EMPTY"
            self.json_mode = True  # 端点不支持 response_format 时自动关掉
            self._async_client = None
            super().__init__(model)

        def load_model(self) -> openai.OpenAI:
            return openai.OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=JUDGE_TIMEOUT,
                                 default_headers=LLM_EXTRA_HEADERS or None)

        def _kwargs(self, prompt: str, schema: type[BaseModel] | None) -> dict[str, Any]:
            kwargs: dict[str, Any] = {"model": self.model_name, "messages": [{"role": "user", "content": prompt}]}
            if JUDGE_TEMPERATURE is not None:
                kwargs["temperature"] = JUDGE_TEMPERATURE
            if schema is not None and self.json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            return kwargs

        def generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
            kwargs = self._kwargs(prompt, schema)
            try:
                resp = self.model.chat.completions.create(**kwargs)
            except openai.BadRequestError:
                if "response_format" not in kwargs:
                    raise
                self.json_mode = False  # 端点不支持 JSON 模式：去掉重试，之后都不再带
                kwargs.pop("response_format")
                resp = self.model.chat.completions.create(**kwargs)
            return _parse(resp.choices[0].message.content or "", schema)

        async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
            if self._async_client is None:
                self._async_client = openai.AsyncOpenAI(base_url=self.base_url, api_key=self.api_key,
                                                        timeout=JUDGE_TIMEOUT, default_headers=LLM_EXTRA_HEADERS or None)
            kwargs = self._kwargs(prompt, schema)
            try:
                resp = await self._async_client.chat.completions.create(**kwargs)
            except openai.BadRequestError:
                if "response_format" not in kwargs:
                    raise
                self.json_mode = False
                kwargs.pop("response_format")
                resp = await self._async_client.chat.completions.create(**kwargs)
            return _parse(resp.choices[0].message.content or "", schema)

        def get_model_name(self) -> str:
            return self.model_name

    class FakeJudge(DeepEvalBaseLLM):
        """仅供自测（JUDGE_FAKE=1）：不联网，按期望答案里的数字 / 文字是否出现在回答里给 9 分或 1 分。"""

        def __init__(self):
            super().__init__("fake-judge")

        def load_model(self) -> "FakeJudge":
            return self

        @staticmethod
        def _section(prompt: str, title: str) -> str:
            m = re.search(rf"{title}:\n(.*?) \n\n", prompt, re.DOTALL)
            return m.group(1) if m else ""

        @staticmethod
        def _numbers(text: str) -> set[float]:
            return {float(n.replace(",", "")) for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text)}

        def generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
            expected, actual = self._section(prompt, "Expected Output"), self._section(prompt, "Actual Output")
            wanted = self._numbers(expected)
            hit = wanted <= self._numbers(actual) if wanted else expected.strip().casefold() in actual.casefold()
            text = json.dumps({"reason": f"（假裁判）期望答案{'出现在' if hit else '没有出现在'}回答里", "score": 9 if hit else 1},
                              ensure_ascii=False)
            return _parse(text, schema)

        async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None) -> Any:
            return self.generate(prompt, schema)

        def get_model_name(self) -> str:
            return "fake-judge"

    def make_judge() -> DeepEvalBaseLLM:
        return FakeJudge() if JUDGE_FAKE else OpenAIJudge(*judge_settings())

    def make_metric(judge: DeepEvalBaseLLM) -> GEval:
        """每题新建一个指标（GEval 实例会记录本次的 score / reason，不能在线程间共用）。"""
        return GEval(name="答案正确性", evaluation_steps=EVALUATION_STEPS,
                     evaluation_params=[_Params.INPUT, _Params.EXPECTED_OUTPUT, _Params.ACTUAL_OUTPUT],
                     threshold=PASS_THRESHOLD, model=judge, async_mode=False, verbose_mode=False)

    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"输入文件 : {INPUT_PATH}")
    print(f"输出目录 : {EVAL_DIR}")
    print("裁判模型 : " + ("fake-judge（JUDGE_FAKE=1，仅供自测）" if JUDGE_FAKE else
                          f"{_judge_model or '（未配置）'}，地址 {_judge_base_url or '（未设，用 OpenAI 官方）'}"))  # 不打印 key
    print(f"通过阈值 : {PASS_THRESHOLD}，并发 {JUDGE_CONCURRENCY}，单次超时 {JUDGE_TIMEOUT:g}s")
    print(f"共 {len(eval_inputs)} 题：已有结论 {len(eval_done)}，本次处理 {len(eval_todo)}"
          f"（送裁判 {len(eval_todo) - eval_n_skip}，不评测 {eval_n_skip}）")

In [ ]:
# ───────────── 步骤 7：逐题评测（JUDGE_CONCURRENCY 道题同时评；每题完成立即追加写入 eval_results.jsonl）─────────────
# 单题出错（如超时、裁判输出无法解析）只记 eval_status=error，不中断整批，下次运行会重评。
if EVAL_SKIP_REASON is None:
    eval_lock = threading.Lock()
    eval_finished = [0]
    eval_judge = make_judge() if len(eval_todo) > eval_n_skip else None  # 全是不评测的题时不需要模型
    JUDGE_NAME = eval_judge.get_model_name() if eval_judge else None

    def evaluate_one(r: dict[str, Any]) -> None:
        t_start = time.perf_counter()
        row = {"id": str(r.get("id")), "question": r.get("question"), "expected": r.get("expected"),
               "answer": r.get("answer"), "doc": r.get("doc"), "eval_score": None, "eval_pass": None,
               "eval_reason": None, "eval_status": "not_evaluated", "judge_model": None, "elapsed_s": 0.0}
        reason = eval_skip_reason(r)
        if reason:
            row["eval_reason"] = reason
        else:
            row["judge_model"] = JUDGE_NAME
            try:
                metric = make_metric(eval_judge)
                metric.measure(LLMTestCase(input=str(r.get("question") or ""), actual_output=str(r.get("answer") or ""),
                                           expected_output=str(r.get("expected"))), _show_indicator=False)
                row.update(eval_score=round(float(metric.score), 4), eval_pass=bool(metric.success),
                           eval_reason=metric.reason, eval_status="evaluated")
            except Exception as exc:  # noqa: BLE001 - 单题失败只记下来，整批继续
                row.update(eval_reason=f"{type(exc).__name__}: {exc}", eval_status="error")
            row["elapsed_s"] = round(time.perf_counter() - t_start, 2)
        with eval_lock:  # 每题完成立即追加写入
            with EVAL_RESULTS.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
            eval_finished[0] += 1
            status = row["eval_status"]
            verdict = (f"{row['eval_score']:.2f} {'通过' if row['eval_pass'] else '不通过'}" if status == "evaluated" else
                       f"ERROR {row['eval_reason']}" if status == "error" else f"不评测：{row['eval_reason']}")
            print(f"[{eval_finished[0]}/{len(eval_todo)}] {row['id']}  {verdict}  {row['elapsed_s']:.1f}s", flush=True)

    eval_t0 = time.time()
    if eval_todo:
        with ThreadPoolExecutor(max_workers=max(1, JUDGE_CONCURRENCY)) as eval_pool:
            list(eval_pool.map(evaluate_one, eval_todo))
    print(f"本次处理 {len(eval_todo)} 题，用时 {time.time() - eval_t0:.1f}s")

In [ ]:
# ───────────── 步骤 7：汇总与导出（eval_summary.md、eval.csv、eval.xlsx）─────────────
# 按 results.jsonl 的题目顺序读出每题最新的评测结果。准确率 = 通过数 / 已评测数（不含未评测和报错的题）。
if EVAL_SKIP_REASON is None:
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

    eval_latest = read_jsonl(EVAL_RESULTS)
    eval_rows = [eval_latest[k] for k in eval_inputs if k in eval_latest]
    eval_evaluated = [r for r in eval_rows if r["eval_status"] == "evaluated"]
    eval_passed = sum(1 for r in eval_evaluated if r["eval_pass"])
    eval_n_not = sum(1 for r in eval_rows if r["eval_status"] == "not_evaluated")
    eval_n_err = sum(1 for r in eval_rows if r["eval_status"] == "error")
    eval_acc_text = (f"{eval_passed / len(eval_evaluated):.1%}（{eval_passed}/{len(eval_evaluated)}）" if eval_evaluated
                     else "无（没有已评测的题）")
    eval_judges = sorted({r["judge_model"] for r in eval_evaluated if r.get("judge_model")})

    def _md_cell(text: Any, limit: int = 80) -> str:
        s = " ".join(str("" if text is None else text).split()).replace("|", "\\|")
        return s if len(s) <= limit else s[: limit - 1] + "…"

    eval_lines = [
        "# 答案评测（deepeval GEval：答案正确性）", "",
        f"- 时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"- 输入：`{INPUT_PATH}`",
        f"- 裁判模型：{', '.join(eval_judges) or '-'}；通过阈值 {PASS_THRESHOLD}",
        f"- **准确率：{eval_acc_text}**",
        f"- 题数 {len(eval_rows)}：已评测 {len(eval_evaluated)}，未评测 {eval_n_not}，报错 {eval_n_err}", "",
        "| ID | 问题 | 得分 | 通过 | 理由摘要 |", "|---|---|---|---|---|",
    ]
    for er in eval_rows:
        score_text = f"{er['eval_score']:.2f}" if er["eval_score"] is not None else "-"
        pass_text = {"evaluated": "✓" if er["eval_pass"] else "✗", "error": "报错", "not_evaluated": "未评测"}[er["eval_status"]]
        eval_lines.append(f"| {_md_cell(er['id'], 40)} | {_md_cell(er['question'], 60)} | {score_text} | {pass_text} | "
                          f"{_md_cell(er['eval_reason'], 120)} |")
    EVAL_SUMMARY.write_text("\n".join(eval_lines) + "\n", encoding="utf-8")

    eval_df = pd.DataFrame([{**r, "doc": "; ".join(r["doc"]) if isinstance(r.get("doc"), list) else r.get("doc")}
                            for r in eval_rows])
    eval_df.to_csv(EVAL_CSV, index=False, encoding="utf-8-sig")

    def _xlsx_cell(v: Any) -> Any:  # 同步骤 5 的 clean_cell：xlsx 不接受控制字符，单元格最长 32767 字符
        return ILLEGAL_CHARACTERS_RE.sub("", v)[:32767] if isinstance(v, str) else v

    (eval_df.map(_xlsx_cell) if hasattr(eval_df, "map") else eval_df.applymap(_xlsx_cell)).to_excel(EVAL_XLSX, index=False)
    display(eval_df)

    print(f"准确率 : {eval_acc_text}；未评测 {eval_n_not}，报错 {eval_n_err}，共 {len(eval_rows)} 题")
    print("产出文件：")
    for eval_file in (EVAL_RESULTS, EVAL_SUMMARY, EVAL_CSV, EVAL_XLSX):
        print(f"  {eval_file}")